# 🎬 Promo con personajes IA: tus avatares furros explicando The Republic of Furrys

Dos de tus avatares (los modelos de Roblox, **sin huesos**: el cuaderno les crea el esqueleto solo) actúan una escena **controlados por IA**:
- **HY-Motion 1.0** (Tencent) genera el movimiento humano real a partir de texto ("una persona se sienta", "truena los dedos"…). Usa la VRAM.
- **Cara 3D**: boca que se abre con la voz y párpados que parpadean · **pelaje real** en cabeza y orejas · miran a quien habla.
- **Voces IA** distintas para cada uno (Kokoro).
- **Historia**: A se burla con el celular («los servers de Discord no sirven») → B: «a ver, siéntate» → lanza `/silla`, aparece una silla y lo **arrastra con las flechas de Roblox Studio** → le explica la república.
- **Escenarios**: cuarto blanco, cuarto negro y lugares hiperrealistas (HDRI) · **cámaras** de seguimiento estabilizada y **POV** · **textos 3D en perspectiva** detrás de ellos.
- **Render** en Blender Cycles con la H100 (la escena usa tu RAM) · **visor en vivo** mientras se renderiza · subtítulos, música y efectos.

**Antes de empezar:** sube los .zip de los modelos a `Anuncio/Personajes IA/`. Luego **Entorno de ejecución → H100 → Ejecutar todo**. El video queda en `Editores - clase 1/Editados/`.

In [ ]:
#@title 1 · Instalar Blender (bpy), voces IA y montar Google Drive
#@markdown **Entorno de ejecución → GPU H100.** La IA de movimiento (HY-Motion) se instala en la celda 7.
#@markdown `CACHE_EN_DRIVE`: guarda lo instalado en tu Drive (`.cache_colab`) para que la próxima vez tarde 1-3 min.
CACHE_EN_DRIVE = True  #@param {type:"boolean"}
import subprocess, sys, os, time
IN_COLAB = 'google.colab' in sys.modules
t_inst = time.time(); CACHE = '/content/drive/MyDrive/.cache_colab'; UV_PY = '/root/.local/share/uv/python'
def pip(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
    if r.returncode != 0: print(f"⚠ No se instaló {' '.join(pkgs)}:", (r.stderr.strip().splitlines() or [''])[-1])
    return r.returncode == 0
def restore_env(name):
    tar = f"{CACHE}/{name}.tar"
    if CACHE_EN_DRIVE and IN_COLAB and os.path.exists(tar) and not os.path.exists(f"/content/{name}/bin/python"):
        t0 = time.time(); r = subprocess.run(['tar', '-xf', tar, '-C', '/'], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"/content/{name}/bin/python"):
            print(f"✓ {name} restaurado desde tu Drive en {time.time() - t0:.0f} s"); return True
    return os.path.exists(f"/content/{name}/bin/python")
def save_env(name, extra=()):
    if CACHE_EN_DRIVE and IN_COLAB and os.path.exists(f"/content/{name}/bin/python"):
        t0 = time.time(); os.makedirs(CACHE, exist_ok=True); tmp = f"/content/{name}.tar"
        parts = [f"content/{name}"] + ([UV_PY.lstrip('/')] if os.path.isdir(UV_PY) else []) + [p.lstrip('/') for p in extra if os.path.exists(p)]
        subprocess.run(['tar', '-cf', tmp, '-C', '/', *parts], capture_output=True); subprocess.run(['cp', tmp, f"{CACHE}/{name}.tar"]); os.remove(tmp)
        print(f"  (guardado en tu Drive para la próxima vez, {time.time() - t0:.0f} s)")
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.Popen(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    pip('uv')
    r = subprocess.run(['uv', 'pip', 'install', '-q', '--system', 'bpy==5.2.3', 'opencv-python-headless', 'soundfile', 'edge-tts', 'num2words', 'pillow'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        for p_ in ('bpy==5.2.3', 'opencv-python-headless', 'soundfile', 'edge-tts', 'num2words'): pip(p_)
    if not restore_env('tts_env'):
        print("Instalando la voz Kokoro (entorno aparte)…")
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/tts_env'], check=False)
        r = subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/tts_env/bin/python', 'kokoro>=0.9.4', 'soundfile',
                            '--extra-index-url', 'https://download.pytorch.org/whl/cpu', '--index-strategy', 'unsafe-best-match'], capture_output=True, text=True)
        if r.returncode != 0: print("⚠ Kokoro no se instaló (se usará la voz Edge):", r.stderr[-300:])
        else: save_env('tts_env')
print(f"Listo en {(time.time() - t_inst) / 60:.1f} min")
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout if os.path.exists('/usr/bin/nvidia-smi') else 'Sin GPU')

In [ ]:
#@title 2 · Configuración de la promo con personajes IA
import os, re, json, math, glob, shutil, zipfile, base64, time, subprocess, sys
import numpy as np, cv2
#@markdown **Carpeta del proyecto** y subcarpeta donde subiste los modelos (los .zip de Roblox: .obj + textura .png).
PROYECTO = '/content/drive/MyDrive/Anuncio'  #@param {type:"string"}
CARPETA_PERSONAJES = "Personajes IA"  #@param {type:"string"}
CARPETA_SALIDA = "Editores - clase 1/Editados"  #@param {type:"string"}
#@markdown **Personaje A** (el del celular, que se burla) y **B** (el que lanza el comando y explica). Parte del nombre del archivo/zip:
MODELO_A = "gatorojo"  #@param {type:"string"}
MODELO_B = "modelo1"  #@param {type:"string"}
SERVER = "The Republic of Furrys"  #@param {type:"string"}
DOMINIO = "therepublicoffury.web.app"  #@param {type:"string"}
#@markdown ---
#@markdown **Voces IA** (Kokoro): `em_alex`, `em_santa` (hombres) o `ef_dora` (mujer).
VOZ_A = "em_santa"  #@param ["em_santa", "em_alex", "ef_dora"]
VOZ_B = "em_alex"  #@param ["em_alex", "em_santa", "ef_dora"]
VELOCIDAD_VOZ = 1.05  #@param {type:"slider", min:0.85, max:1.25, step:0.05}
#@markdown **IA de movimiento** (cómo se mueven): `HY-Motion-1.0` (la mejor, ~26 GB de VRAM), `Lite` (más rápida) o `procedural` (sin IA, para probar).
IA_MOVIMIENTO = "HY-Motion-1.0"  #@param ["HY-Motion-1.0", "HY-Motion-1.0-Lite", "procedural"]
#@markdown ---
PELAJE = True  #@param {type:"boolean"}
#@markdown Escenarios hiperrealistas (HDRI gratis de Poly Haven, CC0). Nombres de https://polyhaven.com/hdris
HDRI_PLAZA = "royal_esplanade, pedestrian_overpass, urban_street_01"  #@param {type:"string"}
HDRI_BOSQUE = "forest_slope, misty_pines, kloofendal_48d_partly_cloudy"  #@param {type:"string"}
#@markdown **Calidad del render final** (en la H100: borrador ~10 min, alta ~45-90 min para ~35 s).
CALIDAD = "alta"  #@param ["borrador", "media", "alta"]
SUBTITULOS = True  #@param {type:"boolean"}
VOLUMEN_MUSICA = 0.45  #@param {type:"slider", min:0, max:1, step:0.05}
GUARDAR_BLEND = True  #@param {type:"boolean"}
IN_COLAB = 'google.colab' in sys.modules
WORK = '/content/promo_ia' if IN_COLAB else os.path.abspath('promo_ia')
CODE = os.path.join(WORK, 'code'); os.makedirs(CODE, exist_ok=True); sys.path.insert(0, CODE)
def _n(s): return re.sub(r'\s+', ' ', s.strip().lower())
def find_dir(rel, base=None):
    cur = base or PROYECTO
    for part in [p for p in re.split(r'[\\/]', rel) if p.strip()]:
        ex = os.path.join(cur, part)
        if os.path.isdir(ex): cur = ex; continue
        c = [d for d in (os.listdir(cur) if os.path.isdir(cur) else []) if os.path.isdir(os.path.join(cur, d)) and _n(d) == _n(part)]
        if not c: return ex
        cur = os.path.join(cur, c[0])
    return cur
OUT_DIR = find_dir(CARPETA_SALIDA); os.makedirs(OUT_DIR, exist_ok=True)
QUAL = {'borrador': (540, 960, 24), 'media': (1080, 1920, 48), 'alta': (1080, 1920, 128)}[CALIDAD]
FPS = 30
print("Proyecto:", PROYECTO, "| salida:", OUT_DIR, "| render:", QUAL)

In [ ]:
#@title 3 · Código del editor 3D (esqueleto, cara, cámaras, escena de Blender) — sólo ejecútala
MODULOS = {'rig': '# Rig automático para avatares R15 de Roblox exportados como OBJ (15 piezas sin huesos) + retarget de movimiento SMPL/SMPL-H.\nimport numpy as np, re, os\ndef load_obj(path):\n    V, VT, groups, cur, mtl = [], [], {}, None, None\n    for l in open(path, encoding=\'utf-8\', errors=\'ignore\'):\n        if l.startswith(\'v \'): V.append(list(map(float, l.split()[1:4])))\n        elif l.startswith(\'vt \'): VT.append(list(map(float, l.split()[1:3])))\n        elif l.startswith(\'mtllib\'): mtl = l.split(maxsplit=1)[1].strip()\n        elif l.startswith(\'g \') or l.startswith(\'o \'):\n            cur = l.split(maxsplit=1)[1].strip(); groups.setdefault(cur, [])\n        elif l.startswith(\'f \'):\n            if cur is None: cur = \'mesh\'; groups.setdefault(cur, [])\n            idx = [tuple(int(x) if x else 0 for x in (p.split(\'/\') + [\'0\', \'0\'])[:2]) for p in l.split()[1:]]\n            groups[cur].append(idx)\n    return np.array(V, float), np.array(VT, float), groups, mtl\n\ndef ry180(p): q = p.copy(); q[..., 0] *= -1; q[..., 2] *= -1; return q    # Roblox mira a -Z → lo giro para que mire a +Z (como SMPL)\n\nR15 = [\'LowerTorso\', \'UpperTorso\', \'Head\', \'LeftUpperArm\', \'LeftLowerArm\', \'LeftHand\', \'RightUpperArm\', \'RightLowerArm\', \'RightHand\',\n       \'LeftUpperLeg\', \'LeftLowerLeg\', \'LeftFoot\', \'RightUpperLeg\', \'RightLowerLeg\', \'RightFoot\']\nPARENT = {\'LowerTorso\': None, \'UpperTorso\': \'LowerTorso\', \'Head\': \'UpperTorso\', \'LeftUpperArm\': \'UpperTorso\', \'LeftLowerArm\': \'LeftUpperArm\',\n          \'LeftHand\': \'LeftLowerArm\', \'RightUpperArm\': \'UpperTorso\', \'RightLowerArm\': \'RightUpperArm\', \'RightHand\': \'RightLowerArm\',\n          \'LeftUpperLeg\': \'LowerTorso\', \'LeftLowerLeg\': \'LeftUpperLeg\', \'LeftFoot\': \'LeftLowerLeg\', \'RightUpperLeg\': \'LowerTorso\',\n          \'RightLowerLeg\': \'RightUpperLeg\', \'RightFoot\': \'RightLowerLeg\'}\n\ndef classify(V, groups):\n    """Identifica cada pieza R15 por su posición (las exportaciones de Roblox no traen nombres útiles)."""\n    P = ry180(V); info = {}\n    for g, faces in groups.items():\n        vi = np.unique([a[0] - 1 for f in faces for a in f]); p = P[vi]\n        info[g] = {\'vi\': vi, \'min\': p.min(0), \'max\': p.max(0), \'c\': p.mean(0), \'n\': len(faces)}\n    ymin, ymax = P[:, 1].min(), P[:, 1].max(); H = ymax - ymin; cx = (P[:, 0].min() + P[:, 0].max()) / 2\n    names = {}\n    rest = dict(info)\n    head = max(rest, key=lambda g: rest[g][\'max\'][1] + 0.001 * rest[g][\'n\']); names[head] = \'Head\'; rest.pop(head)\n    # piernas: piezas cuyo centro está en la mitad de abajo; brazos: piezas laterales arriba; torso: centradas\n    mid = [g for g in rest if rest[g][\'c\'][1] > ymin + 0.3 * H]\n    torso = sorted(mid, key=lambda g: abs(rest[g][\'c\'][0] - cx))[:2]               # las 2 piezas más centradas a media altura\n    tx0 = min(rest[g][\'min\'][0] for g in torso); tx1 = max(rest[g][\'max\'][0] for g in torso)\n    low_t = min(rest[g][\'c\'][1] for g in torso)\n    legs = [g for g in rest if g not in torso and rest[g][\'c\'][1] < low_t and tx0 - 0.02 * H <= rest[g][\'c\'][0] <= tx1 + 0.02 * H]\n    arms = [g for g in rest if g not in torso and g not in legs]\n    if len(legs) != 6 or len(arms) != 6 or len(torso) != 2:\n        raise RuntimeError(f"No pude identificar las 15 piezas (piernas {len(legs)}, brazos {len(arms)}, torso {len(torso)})")\n    lo, up = sorted(torso, key=lambda g: rest[g][\'c\'][1]); names[lo] = \'LowerTorso\'; names[up] = \'UpperTorso\'\n    for side_list, kind in ((legs, \'Leg\'), (arms, \'Arm\')):\n        for side, sel in ((\'Left\', [g for g in side_list if rest[g][\'c\'][0] > cx]), (\'Right\', [g for g in side_list if rest[g][\'c\'][0] <= cx])):\n            if kind == \'Leg\':\n                order = sorted(sel, key=lambda g: -rest[g][\'c\'][1]); labels = [\'UpperLeg\', \'LowerLeg\', \'Foot\']\n            else:   # del hombro a la mano: por distancia al centro del cuerpo (brazos en A)\n                order = sorted(sel, key=lambda g: abs(rest[g][\'c\'][0] - cx) - 0.3 * rest[g][\'c\'][1]); labels = [\'UpperArm\', \'LowerArm\', \'Hand\']\n            for g, lab in zip(order, labels): names[g] = side + lab\n    return names, info, P\n\ndef joints_r15(names, info, P):\n    """Posiciones de articulación (en reposo) a partir de dónde se tocan las piezas."""\n    by = {v: k for k, v in names.items()}\n    def pts(n): return P[info[by[n]][\'vi\']]\n    def contact(a, b):\n        A, B = pts(a), pts(b)\n        # punto medio de los vértices de A más cercanos a B\n        from numpy.linalg import norm\n        cb = B.mean(0); d = norm(A - cb, axis=1); near = A[d <= np.percentile(d, 8)]\n        ca = A.mean(0); d2 = norm(B - ca, axis=1); nearb = B[d2 <= np.percentile(d2, 8)]\n        return (near.mean(0) + nearb.mean(0)) / 2\n    J = {}\n    J[\'LowerTorso\'] = pts(\'LowerTorso\').mean(0)\n    J[\'UpperTorso\'] = contact(\'LowerTorso\', \'UpperTorso\')\n    J[\'Head\'] = contact(\'UpperTorso\', \'Head\')\n    for s in (\'Left\', \'Right\'):\n        J[s + \'UpperArm\'] = contact(\'UpperTorso\', s + \'UpperArm\'); J[s + \'LowerArm\'] = contact(s + \'UpperArm\', s + \'LowerArm\')\n        J[s + \'Hand\'] = contact(s + \'LowerArm\', s + \'Hand\')\n        J[s + \'UpperLeg\'] = contact(\'LowerTorso\', s + \'UpperLeg\'); J[s + \'LowerLeg\'] = contact(s + \'UpperLeg\', s + \'LowerLeg\')\n        J[s + \'Foot\'] = contact(s + \'LowerLeg\', s + \'Foot\')\n    # puntas (para la dirección del último hueso)\n    J[\'_HeadTop\'] = np.array([J[\'Head\'][0], pts(\'Head\')[:, 1].max(), J[\'Head\'][2]])\n    for s in (\'Left\', \'Right\'):\n        h = pts(s + \'Hand\'); J[\'_\' + s + \'HandTip\'] = h[np.argmax(np.linalg.norm(h - J[s + \'Hand\'], axis=1))]\n        f = pts(s + \'Foot\'); J[\'_\' + s + \'Toe\'] = np.array([f[:, 0].mean(), f[:, 1].min() + 0.05, f[:, 2].max()])\n    return J\n\n# ---- SMPL (22 articulaciones del cuerpo) ----\nSMPL_PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19]\nSMPL_NAMES = [\'Pelvis\', \'L_Hip\', \'R_Hip\', \'Spine1\', \'L_Knee\', \'R_Knee\', \'Spine2\', \'L_Ankle\', \'R_Ankle\', \'Spine3\', \'L_Foot\', \'R_Foot\',\n              \'Neck\', \'L_Collar\', \'R_Collar\', \'Head\', \'L_Shoulder\', \'R_Shoulder\', \'L_Elbow\', \'R_Elbow\', \'L_Wrist\', \'R_Wrist\']\n# pieza R15 → (articulación SMPL que la mueve, articulación SMPL hacia donde apunta)\nMAP = {\'LowerTorso\': (\'Pelvis\', \'Spine1\'), \'UpperTorso\': (\'Spine3\', \'Neck\'), \'Head\': (\'Head\', None),\n       \'LeftUpperArm\': (\'L_Shoulder\', \'L_Elbow\'), \'LeftLowerArm\': (\'L_Elbow\', \'L_Wrist\'), \'LeftHand\': (\'L_Wrist\', None),\n       \'RightUpperArm\': (\'R_Shoulder\', \'R_Elbow\'), \'RightLowerArm\': (\'R_Elbow\', \'R_Wrist\'), \'RightHand\': (\'R_Wrist\', None),\n       \'LeftUpperLeg\': (\'L_Hip\', \'L_Knee\'), \'LeftLowerLeg\': (\'L_Knee\', \'L_Ankle\'), \'LeftFoot\': (\'L_Ankle\', \'L_Foot\'),\n       \'RightUpperLeg\': (\'R_Hip\', \'R_Knee\'), \'RightLowerLeg\': (\'R_Knee\', \'R_Ankle\'), \'RightFoot\': (\'R_Ankle\', \'R_Foot\')}\nR15_TIP = {\'LowerTorso\': \'UpperTorso\', \'UpperTorso\': \'Head\', \'Head\': \'_HeadTop\', \'LeftUpperArm\': \'LeftLowerArm\', \'LeftLowerArm\': \'LeftHand\',\n           \'LeftHand\': \'_LeftHandTip\', \'RightUpperArm\': \'RightLowerArm\', \'RightLowerArm\': \'RightHand\', \'RightHand\': \'_RightHandTip\',\n           \'LeftUpperLeg\': \'LeftLowerLeg\', \'LeftLowerLeg\': \'LeftFoot\', \'LeftFoot\': \'_LeftToe\', \'RightUpperLeg\': \'RightLowerLeg\',\n           \'RightLowerLeg\': \'RightFoot\', \'RightFoot\': \'_RightToe\'}\nSMPL_TIP_DEFAULT = {\'Head\': np.array([0, 1, 0]), \'L_Wrist\': np.array([1, 0, 0]), \'R_Wrist\': np.array([-1, 0, 0])}\n\ndef aa2mat(aa):\n    aa = np.asarray(aa, float); th = np.linalg.norm(aa, axis=-1, keepdims=True); k = aa / np.maximum(th, 1e-9)\n    K = np.zeros(aa.shape[:-1] + (3, 3)); kx, ky, kz = k[..., 0], k[..., 1], k[..., 2]\n    K[..., 0, 1], K[..., 0, 2], K[..., 1, 0], K[..., 1, 2], K[..., 2, 0], K[..., 2, 1] = -kz, ky, kz, -kx, -ky, kx\n    th = th[..., None]; I = np.eye(3)\n    return I + np.sin(th) * K + (1 - np.cos(th)) * (K @ K)\ndef align(a, b):\n    """Rotación mínima que lleva la dirección a a la dirección b."""\n    a = a / np.linalg.norm(a); b = b / np.linalg.norm(b); v = np.cross(a, b); c = float(np.dot(a, b))\n    if c < -0.9999:\n        ax = np.cross(a, [1, 0, 0]); ax = ax if np.linalg.norm(ax) > 1e-3 else np.cross(a, [0, 1, 0])\n        return aa2mat(ax / np.linalg.norm(ax) * np.pi)\n    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]]); return np.eye(3) + K + K @ K / (1 + c)\n\ndef global_rots(poses):\n    """poses (T, >=66) eje-ángulo SMPL → rotaciones globales (T, 22, 3, 3)."""\n    T = poses.shape[0]; R = aa2mat(poses[:, :66].reshape(T, 22, 3)); G = np.zeros_like(R)\n    for j in range(22):\n        G[:, j] = R[:, j] if SMPL_PARENTS[j] < 0 else G[:, SMPL_PARENTS[j]] @ R[:, j]\n    return G\n\ndef fk_smpl(G, rest):\n    """Posiciones de las articulaciones SMPL (sin traslación) para rotaciones globales G."""\n    T = G.shape[0]; P = np.zeros((T, 22, 3))\n    for j in range(1, 22):\n        p = SMPL_PARENTS[j]; P[:, j] = P[:, p] + np.einsum(\'tij,j->ti\', G[:, p], rest[j] - rest[p])\n    return P\n\ndef retarget(poses, trans, smpl_rest, J, foot_pts=None, absolute=False):\n    """Movimiento SMPL → por cuadro, rotación (3x3) y posición de la articulación de cada pieza R15\n    (marco y-arriba, el avatar mira a +Z). La malla en reposo tiene rotación identidad: v = R·(v_reposo − J_pieza) + pos."""\n    T = poses.shape[0]; G = global_rots(poses); idx = {n: i for i, n in enumerate(SMPL_NAMES)}\n    smpl_hip = smpl_rest[idx[\'Pelvis\'], 1] - min(smpl_rest[idx[\'L_Ankle\'], 1], smpl_rest[idx[\'R_Ankle\'], 1])\n    r15_hip = J[\'LowerTorso\'][1] - min(J[\'LeftFoot\'][1], J[\'RightFoot\'][1])\n    scale = r15_hip / max(1e-6, smpl_hip)\n    C = {}\n    for part, (sj, st) in MAP.items():\n        r = J[R15_TIP[part]] - J[part]\n        w = (smpl_rest[idx[st]] - smpl_rest[idx[sj]]) if st else SMPL_TIP_DEFAULT.get(sj, np.array([0., 1., 0.]))\n        C[part] = align(r, w)\n    rot = {p: G[:, idx[MAP[p][0]]] @ C[p] for p in R15}\n    pelvis_w = smpl_rest[0][None] + trans                       # pelvis en el mundo SMPL\n    o = np.zeros(3) if absolute else pelvis_w[0]\n    root = np.zeros((T, 3)); root[:, 0] = (pelvis_w[:, 0] - o[0]) * scale; root[:, 2] = (pelvis_w[:, 2] - o[2]) * scale\n    root[:, 1] = pelvis_w[:, 1] * scale\n    pos = {\'LowerTorso\': root}\n    for p in R15[1:]:\n        par = PARENT[p]; pos[p] = pos[par] + np.einsum(\'tij,j->ti\', rot[par], J[p] - J[par])\n    # apoyar en el piso: el punto más bajo de los pies (en todo el clip) toca y=0\n    lows = []\n    for f in (\'LeftFoot\', \'RightFoot\'):\n        pts = foot_pts[f] if foot_pts is not None else np.array([J[f]])\n        lows.append((np.einsum(\'tij,nj->tni\', rot[f], pts - J[f]) + pos[f][:, None]).min(1)[:, 1])\n    ground = np.min(np.stack(lows)) if T else 0.0\n    for p in pos: pos[p] = pos[p] - [0, ground, 0]\n    return rot, pos, scale\n\ndef rest_pose_motion(T=1):\n    return np.zeros((T, 156)), np.zeros((T, 3))\n\n# Plantilla aproximada de SMPL en T (sólo para pruebas locales; en Colab se usa j_template.bin de HY-Motion)\nSMPL_REST_APPROX = np.array([\n    [0, .93, 0], [.06, .84, 0], [-.06, .84, 0], [0, 1.04, 0], [.1, .45, 0], [-.1, .45, 0], [0, 1.17, 0], [.09, .07, -.03], [-.09, .07, -.03],\n    [0, 1.23, 0], [.11, .02, .1], [-.11, .02, .1], [0, 1.45, 0], [.08, 1.37, 0], [-.08, 1.37, 0], [0, 1.52, .03], [.18, 1.4, 0], [-.18, 1.4, 0],\n    [.44, 1.4, 0], [-.44, 1.4, 0], [.69, 1.4, 0], [-.69, 1.4, 0]], float)\n\ndef head_front_image(V, VT, groups, names, P, tex, S=600):\n    """Vista frontal ortográfica de la cabeza con textura + profundidad (para encontrar ojos, nariz y boca)."""\n    import cv2\n    head = [g for g, n in names.items() if n == \'Head\'][0]\n    hv = P[[a[0] - 1 for f in groups[head] for a in f]]; mn, mx = hv.min(0), hv.max(0); sc = S * 0.9 / (mx - mn)[:2].max()\n    img = np.zeros((S, S, 3), np.uint8); depth = np.full((S, S), -1e9, np.float32); tris = []\n    for f in groups[head]:\n        vi = [a[0] - 1 for a in f]; ti = [a[1] - 1 for a in f]\n        for k in range(1, len(f) - 1):\n            ids = [0, k, k + 1]; p = P[[vi[i] for i in ids]]; uv = VT[[ti[i] for i in ids]].mean(0)\n            x = (p[:, 0] - mn[0]) * sc + S * 0.05; y = S - ((p[:, 1] - mn[1]) * sc + S * 0.05)\n            c = tex[int(np.clip(1 - uv[1], 0, 1) * (tex.shape[0] - 1)), int(np.clip(uv[0], 0, 1) * (tex.shape[1] - 1))]\n            if tex.shape[2] == 4 and c[3] < 40: continue\n            tris.append((p[:, 2].mean(), np.stack([x, y], 1).astype(np.int32), tuple(int(v) for v in c[:3])))\n    for z, pts, c in sorted(tris, key=lambda t: t[0]):\n        cv2.fillConvexPoly(img, pts, c); cv2.fillConvexPoly(depth, pts, float(z))\n    def to3d(x, y):\n        X = (x - S * 0.05) / sc + mn[0]; Y = (S - y - S * 0.05) / sc + mn[1]\n        yy, xx = int(np.clip(y, 0, S - 1)), int(np.clip(x, 0, S - 1)); win = depth[max(0, yy - 3):yy + 4, max(0, xx - 3):xx + 4]\n        return np.array([X, Y, float(win.max())])\n    return img, depth, to3d, sc\n\ndef face_features(V, VT, groups, names, P, J, tex, S=600):\n    """Ojos, nariz y boca dibujados en la textura → posiciones 3D sobre la superficie de la cara."""\n    import cv2\n    img, depth, to3d, sc = head_front_image(V, VT, groups, names, P, tex, S)\n    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV); valid = depth > -1e8\n    dark = ((hsv[..., 2] < 70) & valid).astype(np.uint8)\n    blob = cv2.dilate(dark, np.ones((15, 15), np.uint8))\n    n, lab, st, cen = cv2.connectedComponentsWithStats(blob)\n    cx = S / 2; comps = [(i, st[i]) for i in range(1, n) if st[i, cv2.CC_STAT_AREA] > S * S * 0.0015]\n    best, pair = -1e9, None\n    for a in range(len(comps)):\n        for b in range(a + 1, len(comps)):\n            ia, sa = comps[a]; ib, sb = comps[b]\n            ca, cb = cen[ia], cen[ib]\n            if (ca[0] - cx) * (cb[0] - cx) >= 0: continue                    # uno a cada lado\n            sym = abs(abs(ca[0] - cx) - abs(cb[0] - cx)) + abs(ca[1] - cb[1])\n            score = sa[cv2.CC_STAT_AREA] + sb[cv2.CC_STAT_AREA] - 40 * sym\n            if score > best: best, pair = score, (ia, ib)\n    out = {}\n    if pair:\n        for i in pair:\n            x0, y0, w, h = st[i, :4]; c = cen[i]; side = \'L\' if c[0] > cx else \'R\'     # +x = izquierda del personaje\n            out[\'eye_\' + side] = (to3d(c[0], y0 + h / 2), (h / 2) / sc, (w / 2) / sc)\n        eye_y = np.mean([cen[i][1] for i in pair]); eye_h = np.mean([st[i, 3] for i in pair])\n        # nariz: mancha oscura chica, centrada, debajo de los ojos\n        nose = [cen[i] for i in range(1, n) if i not in pair and abs(cen[i][0] - cx) < S * 0.08 and eye_y < cen[i][1] < eye_y + S * 0.3]\n        ny = min(nose, key=lambda c: c[1])[1] if nose else eye_y + eye_h * 0.9\n        my = ny + 0.55 * (ny - eye_y)\n        mx_ = np.mean([cen[i][0] for i in pair]); out[\'mouth\'] = (to3d(mx_, my), (eye_h * 0.28) / sc)\n        ring = (np.abs(np.arange(S)[:, None] - (eye_y - eye_h * 0.9)) < 6) & (np.abs(np.arange(S)[None] - cx) < S * 0.12) & valid\n        skin = img[ring & (dark == 0)]\n        out[\'skin\'] = (np.median(skin, 0)[::-1] / 255) if len(skin) else np.array([0.9, 0.85, 0.8])\n    else:\n        hh = P[:, 1].max() - J[\'Head\'][1]; hz = P[:, 2].max()\n        out[\'eye_L\'] = (np.array([J[\'Head\'][0] + 0.18 * hh, J[\'Head\'][1] + 0.5 * hh, hz]), 0.1 * hh, 0.1 * hh)\n        out[\'eye_R\'] = (np.array([J[\'Head\'][0] - 0.18 * hh, J[\'Head\'][1] + 0.5 * hh, hz]), 0.1 * hh, 0.1 * hh)\n        out[\'mouth\'] = (np.array([J[\'Head\'][0], J[\'Head\'][1] + 0.25 * hh, hz]), 0.06 * hh); out[\'skin\'] = np.array([0.9, 0.85, 0.8])\n    out[\'front_img\'] = img\n    return out\n', 'movimiento': '# Movimiento: clips de la IA (HY-Motion, formato SMPL-H) o de respaldo procedural → una línea de tiempo por personaje.\nimport numpy as np, math\nimport rig\nFPS = 30\nIDX = {n: i for i, n in enumerate(rig.SMPL_NAMES)}\n\n# ---------- cuaterniones ----------\ndef aa2q(aa):\n    aa = np.asarray(aa, float); th = np.linalg.norm(aa, axis=-1, keepdims=True); k = aa / np.maximum(th, 1e-9)\n    return np.concatenate([np.cos(th / 2), k * np.sin(th / 2)], -1)\ndef q2aa(q):\n    q = q / np.linalg.norm(q, axis=-1, keepdims=True); q = np.where(q[..., :1] < 0, -q, q)\n    th = 2 * np.arccos(np.clip(q[..., :1], -1, 1)); s = np.sqrt(np.maximum(1 - q[..., :1] ** 2, 1e-12))\n    return q[..., 1:] / s * th\ndef slerp(q0, q1, t):\n    d = np.sum(q0 * q1, -1, keepdims=True); q1 = np.where(d < 0, -q1, q1); d = np.abs(d)\n    t = np.asarray(t, float)[..., None] if np.ndim(t) else t\n    lin = d > 0.9995; th = np.arccos(np.clip(d, -1, 1)); s = np.sin(th)\n    w0 = np.where(lin, 1 - t, np.sin((1 - t) * th) / np.where(s == 0, 1, s)); w1 = np.where(lin, t, np.sin(t * th) / np.where(s == 0, 1, s))\n    q = w0 * q0 + w1 * q1; return q / np.linalg.norm(q, axis=-1, keepdims=True)\ndef qmul(a, b):\n    w1, x1, y1, z1 = np.moveaxis(a, -1, 0); w2, x2, y2, z2 = np.moveaxis(b, -1, 0)\n    return np.stack([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,\n                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2], -1)\ndef yaw_q(a): return np.array([math.cos(a / 2), 0, math.sin(a / 2), 0])\n\nclass Clip:\n    def __init__(self, poses, trans):\n        P = np.zeros((len(poses), 156)); P[:, :poses.shape[1]] = poses; self.poses = P; self.trans = np.asarray(trans, float).copy()\n    @property\n    def T(self): return len(self.poses)\n    def facing(self, t=0):\n        R = rig.aa2mat(self.poses[t, :3]); f = R @ np.array([0, 0, 1.0]); return math.atan2(f[0], f[2])\n    def place(self, yaw, xz, at=\'start\'):\n        """Gira el clip para que mire hacia `yaw` (al inicio o al final) y lo pone en la posición xz (de la pelvis)."""\n        t = 0 if at == \'start\' else self.T - 1\n        d = yaw - self.facing(t); q = yaw_q(d); Ry = rig.aa2mat(np.array([0, d, 0]))\n        self.poses[:, :3] = q2aa(qmul(q[None], aa2q(self.poses[:, :3])))\n        self.trans = self.trans @ Ry.T\n        p = self.trans[t]; self.trans[:, 0] += xz[0] - p[0]; self.trans[:, 2] += xz[1] - p[2]\n        return self\n    def fit(self, T):\n        """Ajusta la duración: si sobra se recorta, si falta se sostiene la última pose con una respiración suave."""\n        if self.T >= T: self.poses, self.trans = self.poses[:T], self.trans[:T]; return self\n        n = T - self.T; P = np.repeat(self.poses[-1:], n, 0); Tr = np.repeat(self.trans[-1:], n, 0)\n        tt = np.arange(n) / FPS; P[:, IDX[\'Spine3\'] * 3] += 0.02 * np.sin(2 * np.pi * tt / 4.0)\n        self.poses = np.concatenate([self.poses, P]); self.trans = np.concatenate([self.trans, Tr]); return self\n\ndef concat(clips, blend=8):\n    """Une clips (ya colocados) con mezcla suave de `blend` cuadros."""\n    poses, trans = clips[0].poses.copy(), clips[0].trans.copy()\n    for c in clips[1:]:\n        P, Tr = c.poses.copy(), c.trans.copy()\n        b = min(blend, len(poses), len(P))\n        if b > 0:\n            w = (np.arange(b) + 1) / (b + 1)\n            qa = aa2q(poses[-b:].reshape(b, 52, 3)); qb = aa2q(P[:b].reshape(b, 52, 3))\n            P[:b] = q2aa(slerp(qa, qb, w[:, None])).reshape(b, 156)\n            Tr[:b] = trans[-b:] * (1 - w[:, None]) + Tr[:b] * w[:, None]\n        poses = np.concatenate([poses, P]); trans = np.concatenate([trans, Tr])\n    return Clip(poses, trans)\n\n# ---------- respaldo procedural (si la IA de movimiento no está disponible) ----------\ndef _noise(T, f, seed):\n    r = np.random.default_rng(seed); t = np.arange(T) / FPS\n    return sum(np.sin(2 * np.pi * f * k * t + r.uniform(0, 6)) / k for k in (1, 2.3, 3.7)) / 1.6\ndef procedural(kind, T, seed=0):\n    """Movimientos básicos y naturales (respiración, cambio de peso, gestos) hechos a mano: sólo por si falla la IA."""\n    P = np.zeros((T, 156)); tr = np.zeros((T, 3)); t = np.arange(T) / FPS; tr[:, 1] = 0.0\n    def j(name, axis, v): P[:, IDX[name] * 3 + axis] += v\n    # base: brazos abajo, respiración, balanceo\n    j(\'L_Shoulder\', 2, -1.25); j(\'R_Shoulder\', 2, 1.25); j(\'L_Elbow\', 1, -0.15); j(\'R_Elbow\', 1, 0.15)\n    j(\'Spine3\', 0, 0.03 * np.sin(2 * np.pi * t / 3.8)); j(\'Pelvis\', 2, 0.03 * _noise(T, 0.15, seed)); tr[:, 0] = 0.015 * _noise(T, 0.12, seed + 1)\n    j(\'Head\', 1, 0.12 * _noise(T, 0.2, seed + 2)); j(\'Head\', 0, 0.06 * _noise(T, 0.25, seed + 3))\n    talk = kind.startswith(\'talk\') or kind in (\'point\', \'snap\', \'drag_hand\')\n    if talk:\n        g = np.clip(_noise(T, 0.6, seed + 4), -1, 1)\n        j(\'R_Shoulder\', 2, -0.55 - 0.35 * g); j(\'R_Elbow\', 1, 1.1 + 0.4 * np.abs(g)); j(\'R_Shoulder\', 1, -0.3)\n        j(\'L_Shoulder\', 2, 0.45 + 0.3 * np.clip(_noise(T, 0.5, seed + 5), -1, 1)); j(\'L_Elbow\', 1, -0.9); j(\'L_Shoulder\', 1, 0.25)\n        j(\'Spine3\', 1, 0.08 * g); j(\'Head\', 0, 0.08 * np.sin(2 * np.pi * 2.2 * t))\n    if kind == \'talk_no\':\n        j(\'Head\', 1, 0.35 * np.sin(2 * np.pi * 3 * t) * np.exp(-t)); j(\'R_Shoulder\', 2, -0.8); j(\'R_Elbow\', 1, 1.6)\n    if kind in (\'phone\', \'phone_laugh\'):\n        j(\'R_Shoulder\', 2, -0.85); j(\'R_Shoulder\', 1, -0.5); j(\'R_Elbow\', 1, 1.9); j(\'R_Wrist\', 0, -0.5); j(\'Head\', 0, 0.35); j(\'Head\', 1, -0.15)\n        if kind == \'phone_laugh\':\n            j(\'Spine3\', 0, 0.12 * np.abs(np.sin(2 * np.pi * 4 * t)) * np.exp(-t / 2.5)); j(\'Head\', 0, -0.25 * np.abs(np.sin(2 * np.pi * 4 * t)) * np.exp(-t / 2.5))\n    if kind == \'snap\':\n        up = np.clip(t / 0.4, 0, 1); j(\'R_Shoulder\', 2, -2.0 * up); j(\'R_Elbow\', 1, 0.6 * up)\n    if kind == \'point\':\n        j(\'R_Shoulder\', 1, -1.3); j(\'R_Shoulder\', 2, -0.9); j(\'R_Elbow\', 1, -0.6)\n    if kind == \'drag_hand\':\n        j(\'R_Shoulder\', 1, -1.2 + 0.6 * np.clip(t / 1.5, 0, 1)); j(\'R_Shoulder\', 2, -0.9)\n    if kind == \'flail\':\n        j(\'L_Shoulder\', 2, 1.7 + 0.8 * np.sin(2 * np.pi * 2.5 * t)); j(\'R_Shoulder\', 2, -1.7 + 0.8 * np.sin(2 * np.pi * 2.5 * t + 1))\n        j(\'L_Hip\', 0, -0.3 * np.sin(2 * np.pi * 2 * t)); j(\'R_Hip\', 0, 0.3 * np.sin(2 * np.pi * 2 * t)); j(\'L_Knee\', 0, 0.5); j(\'R_Knee\', 0, 0.5)\n    if kind in (\'sit\', \'sit_idle\', \'sit_shrug\', \'stand_up\'):\n        if kind == \'sit\': s = np.clip(t / 1.0, 0, 1); s = s * s * (3 - 2 * s)\n        elif kind == \'stand_up\': s = 1 - np.clip(t / 0.8, 0, 1); s = s * s * (3 - 2 * s)\n        else: s = np.ones(T)\n        j(\'L_Hip\', 0, -1.55 * s); j(\'R_Hip\', 0, -1.55 * s); j(\'L_Knee\', 0, 1.55 * s); j(\'R_Knee\', 0, 1.55 * s)\n        j(\'Spine1\', 0, 0.12 * s); tr[:, 1] -= 0.42 * s; tr[:, 2] -= 0.12 * s\n        j(\'L_Shoulder\', 2, 0.3 * s); j(\'R_Shoulder\', 2, -0.3 * s); j(\'L_Elbow\', 1, -0.6 * s); j(\'R_Elbow\', 1, 0.6 * s)\n        if kind == \'sit_shrug\':\n            sh = np.clip(np.sin(np.pi * np.clip((t - 0.2) / 1.2, 0, 1)), 0, 1)\n            j(\'L_Collar\', 2, 0.35 * sh); j(\'R_Collar\', 2, -0.35 * sh); j(\'L_Elbow\', 1, -0.9 * sh); j(\'R_Elbow\', 1, 0.9 * sh); j(\'Head\', 2, 0.2 * sh)\n    tr[:, 1] += 0.0\n    return Clip(P, tr)\n\n# ---------- cara: labios con la voz y parpadeos ----------\ndef lipsync(wav, sr, T, t0):\n    """Apertura de boca (0-1) y ancho por cuadro a partir del audio de la voz (envolvente + brillo)."""\n    o = np.zeros(T); w = np.full(T, 0.5); hop = sr // FPS\n    for f in range(len(wav) // hop):\n        g = t0 + f\n        if not (0 <= g < T): continue\n        seg = wav[f * hop:(f + 1) * hop]; rms = float(np.sqrt(np.mean(seg ** 2)) + 1e-9)\n        o[g] = np.clip((20 * np.log10(rms) + 42) / 26, 0, 1)\n        sp = np.abs(np.fft.rfft(seg)); fr = np.fft.rfftfreq(len(seg), 1 / sr)\n        w[g] = np.clip((np.sum(sp * fr) / (np.sum(sp) + 1e-9) - 900) / 2200, 0, 1)\n    o = np.convolve(o, [0.25, 0.5, 0.25], \'same\'); return o, w\ndef blinks(T, seed):\n    r = np.random.default_rng(seed); b = np.zeros(T); t = int(r.uniform(15, 60))\n    while t < T - 5:\n        b[t:t + 5] = [0.5, 1, 1, 0.6, 0.2][:len(b[t:t + 5])]; t += int(r.uniform(60, 150))\n    return b\n\n# ---------- mirada: la cabeza sigue a quien habla ----------\ndef look_at(rot, pos, J, target_fn, weight, max_deg=45):\n    """Gira la cabeza (y un poco el torso) hacia el objetivo. target_fn(t) → punto (marco y-arriba); weight (T,) 0-1."""\n    T = len(pos[\'Head\'])\n    for t in range(T):\n        if weight[t] <= 0: continue\n        head_p = pos[\'Head\'][t] + rot[\'Head\'][t] @ np.array([0, 0.4, 0]) * 0\n        d = target_fn(t) - pos[\'Head\'][t]; d[1] *= 0.5\n        cur = rot[\'Head\'][t] @ np.array([0, 0, 1.0])\n        R = rig.align(cur, d / (np.linalg.norm(d) + 1e-9))\n        ang = math.degrees(math.acos(np.clip(np.dot(cur, d / (np.linalg.norm(d) + 1e-9)), -1, 1)))\n        k = weight[t] * min(1.0, max_deg / max(ang, 1e-6))\n        aa = rig_mat2aa(R) * k\n        rot[\'Head\'][t] = rig.aa2mat(aa) @ rot[\'Head\'][t]\n    return rot\ndef rig_mat2aa(R):\n    c = np.clip((np.trace(R) - 1) / 2, -1, 1); th = math.acos(c)\n    if th < 1e-6: return np.zeros(3)\n    v = np.array([R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1]]) / (2 * math.sin(th)); return v * th\n', 'director': '# Director: guion (beats) + duración de las voces + clips de movimiento → línea de tiempo completa (todo en cuadros a 30 fps).\nimport numpy as np, math\nimport movimiento as mv\nFPS = mv.FPS\n# Marcas en el set (coordenadas de Blender, unidades del avatar): A a la izquierda, B a la derecha, mirándose en 3/4 hacia cámara\nMARK = {\'A\': np.array([-3.0, 0.0]), \'B\': np.array([3.0, 0.0])}\nCHAIR = np.array([-5.6, -1.4])                              # a donde B arrastra a A\ndef yaw_towards(src, dst):\n    d = dst - src; return math.atan2(d[0], -d[1])           # marco SMPL: mirar a la dirección Blender d\ndef world_to_smpl(xy, scale): return np.array([xy[0], -xy[1]]) / scale\ndef smpl_to_world(xz, scale): return np.array([xz[0], -xz[1]]) * scale\n\ndef plan(beats, voice_dur, pad=0.35, min_beat=1.6):\n    """Tiempos de cada beat (cuadros). La voz empieza 0.15 s después del inicio del beat."""\n    f = 0\n    for b in beats:\n        d = b.get(\'dur\') or max(min_beat, voice_dur.get(b[\'id\'], 0) + pad + 0.15)\n        b[\'f0\'] = f; b[\'n\'] = int(round(d * FPS)); b[\'voz_f\'] = f + int(0.15 * FPS); f += b[\'n\']\n    return f\n\ndef build_character(who, beats, get_clip, scale, total, seed=0):\n    """Une los clips del personaje con su posición/orientación en cada beat. get_clip(beat, who) → Clip (de la IA o procedural)."""\n    other = \'B\' if who == \'A\' else \'A\'\n    clips, info = [], {\'drag\': None, \'sit\': None}\n    pos = MARK[who].copy(); seated = False\n    for b in beats:\n        act = b[who]; n = b[\'n\']\n        c = get_clip(b, who).fit(n)\n        facing_pt = MARK[other] if not act.get(\'mira_camara\') else np.array([pos[0] * 0.3, -20.0])\n        yaw = yaw_towards(pos if not seated else CHAIR, facing_pt)\n        if act.get(\'kind\') == \'sit\':                             # termina sentado en la silla\n            c.place(yaw_towards(CHAIR, MARK[other]), world_to_smpl(CHAIR, scale), at=\'end\')\n            info[\'sit\'] = {\'f0\': b[\'f0\'], \'f1\': b[\'f0\'] + n, \'start_xy\': smpl_to_world(c.trans[0, [0, 2]], scale)}\n            seated = True; pos = CHAIR.copy()\n        elif seated and act.get(\'kind\') in (\'sit_idle\', \'sit_shrug\', \'stand_up\'):\n            c.place(yaw_towards(CHAIR, MARK[other]), world_to_smpl(CHAIR, scale), at=\'start\')\n            if act.get(\'kind\') == \'stand_up\':\n                seated = False; pos = smpl_to_world(c.trans[-1, [0, 2]], scale)\n        else:\n            c.place(yaw, world_to_smpl(pos, scale), at=\'start\')\n        if act.get(\'drag\'):\n            info[\'drag\'] = {\'f0\': b[\'f0\'], \'f1\': b[\'f0\'] + n}\n        clips.append(c)\n    full = mv.concat(clips, blend=8)\n    # arrastre con las flechas: A flota del punto de partida a donde empieza a sentarse\n    if info[\'drag\'] and info[\'sit\']:\n        f0, f1 = info[\'drag\'][\'f0\'], info[\'drag\'][\'f1\']; a = world_to_smpl(MARK[who], scale); z = world_to_smpl(info[\'sit\'][\'start_xy\'], scale)\n        for f in range(f0, f1):\n            u = (f - f0) / max(1, f1 - f0 - 1); e = u * u * (3 - 2 * u)\n            full.trans[f, [0, 2]] = a * (1 - e) + z * e\n            full.trans[f, 1] += 0.35 * math.sin(math.pi * min(1, u * 1.15))           # lo levanta en el aire\n        # hasta que empieza a sentarse, queda en el punto final del arrastre\n        s0 = info[\'sit\'][\'f0\']; full.trans[f1:s0, 0] = z[0]; full.trans[f1:s0, 2] = z[1]\n    return full.fit(total), info\n', 'camara': '# Cámara virtual: seguimiento estabilizado (resorte crítico) y POV, en coordenadas de Blender. Una sola cámara animada.\nimport numpy as np, math\nimport escena_math as em\nFPS = 30\ndef head_points(R, P, J, face):\n    """Posición del centro de la cara (entre los ojos) por cuadro, en Blender. R/P (T,15,...) marco y-arriba."""\n    hk = 2   # índice de \'Head\' en rig.R15\n    eyes = (face[\'eye_L\'][0] + face[\'eye_R\'][0]) / 2 - J[\'Head\']\n    pts = P[:, hk] + np.einsum(\'tij,j->ti\', R[:, hk], eyes)\n    fwd = np.einsum(\'tij,j->ti\', R[:, hk], np.array([0, 0, 1.0]))\n    return em.to_b(pts), em.to_b(fwd)\ndef spring(x, omega=4.0):\n    """Suavizado de resorte crítico (como un estabilizador de cámara)."""\n    y = np.zeros_like(x); v = np.zeros_like(x[0]); y[0] = x[0]; dt = 1 / FPS\n    for t in range(1, len(x)):\n        a = omega * omega * (x[t] - y[t - 1]) - 2 * omega * v; v = v + a * dt; y[t] = y[t - 1] + v * dt\n    return y\nSHOTS = {\'cerca\': (8.5, 0.95), \'medio\': (13.0, 0.85), \'amplio\': (34.0, 0.55)}   # distancia, altura relativa (fracción de la altura de la cabeza)\ndef solve(beats, heads, fwds, total, aspect_vertical=True):\n    """beats con \'cam\': \'medio_A\' | \'cerca_B\' | \'amplio\' | \'pov_A\' (desde los ojos de A mirando a B)… → loc (T,3), target (T,3), lens (T,)."""\n    loc = np.zeros((total, 3)); tgt = np.zeros((total, 3)); lens = np.full(total, 32.0)\n    rng = np.random.default_rng(3)\n    for b in beats:\n        f0, f1 = b[\'f0\'], min(total, b[\'f0\'] + b[\'n\']); cam = b.get(\'cam\', \'medio_\' + (b.get(\'habla\') or \'A\'))\n        kind, _, who = cam.partition(\'_\'); n = f1 - f0\n        if n <= 0: continue\n        if kind == \'pov\':\n            other = \'B\' if who == \'A\' else \'A\'\n            eye = spring(heads[who][f0:f1] + fwds[who][f0:f1] * 0.25, 6.0)\n            look = spring(heads[other][f0:f1] + np.array([0, 0, -0.3]), 5.0)\n            hand = np.cumsum(rng.normal(0, 0.004, (n, 3)), 0); hand -= np.linspace(hand[0], hand[-1], n)\n            loc[f0:f1] = eye + hand; tgt[f0:f1] = look; lens[f0:f1] = 24.0\n            continue\n        if kind == \'amplio\':\n            mid = (heads[\'A\'][f0:f1] + heads[\'B\'][f0:f1]) / 2; mid[:, 2] *= 0.75\n            subj = spring(mid, 2.5); d, hgt = SHOTS[\'amplio\']\n            off = np.array([0.0, -d, 0.0]); drift = np.linspace(0, 1, n)[:, None] * np.array([1.2, 1.5, 0.0])     # dolly lento\n            loc[f0:f1] = subj + off + drift * 3 + np.array([0, 0, -1.5]); tgt[f0:f1] = subj + np.array([0, 0, -2.4]); lens[f0:f1] = 28.0\n            continue\n        other = \'B\' if who == \'A\' else \'A\'\n        subj = spring(heads[who][f0:f1], 3.0); d, hgt = SHOTS.get(kind, SHOTS[\'medio\'])\n        # desde el lado del otro personaje, en 3/4 (sobre el hombro, sin taparlo)\n        to_other = heads[other][f0] - heads[who][f0]; to_other[2] = 0; to_other /= (np.linalg.norm(to_other) + 1e-9)\n        front = np.array([0, -1.0, 0]); dirc = front * 0.55 + to_other * 0.7; dirc /= np.linalg.norm(dirc)\n        drift = np.linspace(-0.5, 0.5, n)[:, None] * np.cross(dirc, [0, 0, 1]) * 0.6 + np.linspace(0.25, 0, n)[:, None] * dirc * d\n        hand = np.cumsum(rng.normal(0, 0.003, (n, 3)), 0); hand -= np.linspace(hand[0], hand[-1], n)\n        loc[f0:f1] = subj + dirc * d + drift + np.array([0, 0, -0.06 * d]) + hand\n        tgt[f0:f1] = subj + np.array([0, 0, -1.6 if kind == \'medio\' else -0.6]); lens[f0:f1] = 32.0 if kind == \'medio\' else 40.0\n    return loc, tgt, lens\n', 'escena_math': 'import numpy as np\nA = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], float)\ndef to_b(p): return np.asarray(p) @ A.T\n', 'escena': '# Construcción de la escena en Blender (bpy): avatares por piezas, materiales, animación por cuadro.\nimport bpy, bmesh, math, os, numpy as np\nfrom mathutils import Matrix, Vector, Quaternion\nimport rig\nA = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], float)          # y-arriba (SMPL) → z-arriba (Blender); +Z SMPL → −Y Blender\ndef to_b(p): return np.asarray(p) @ A.T\ndef rot_b(R): return A @ R @ A.T\n\ndef clear_scene():\n    bpy.ops.wm.read_factory_settings(use_empty=True)\n\ndef material_tex(name, tex_path):\n    m = bpy.data.materials.new(name); m.use_nodes = True; nt = m.node_tree; b = nt.nodes[\'Principled BSDF\']\n    img = bpy.data.images.load(tex_path); t = nt.nodes.new(\'ShaderNodeTexImage\'); t.image = img; t.interpolation = \'Closest\'\n    nt.links.new(t.outputs[\'Color\'], b.inputs[\'Base Color\']); nt.links.new(t.outputs[\'Alpha\'], b.inputs[\'Alpha\'])\n    b.inputs[\'Roughness\'].default_value = 0.62\n    try: b.inputs[\'Sheen Weight\'].default_value = 0.35; b.inputs[\'Sheen Roughness\'].default_value = 0.4    # brillo suave de tela/pelaje\n    except KeyError: pass\n    try: m.surface_render_method = \'DITHERED\'\n    except Exception: pass\n    return m\n\nclass Avatar:\n    def __init__(self, name, obj_path, tex_path):\n        V, VT, G, _ = rig.load_obj(obj_path)\n        self.names, info, P = rig.classify(V, G); self.J = rig.joints_r15(self.names, info, P)\n        self.name = name; self.objs = {}; self.P = P\n        mat = material_tex(name + \'_mat\', tex_path)\n        self.foot_pts = {}\n        for g, faces in G.items():\n            part = self.names[g]\n            vi = sorted({a[0] - 1 for f in faces for a in f}); remap = {v: k for k, v in enumerate(vi)}\n            verts = to_b(P[vi] - self.J[part])                       # origen = articulación\n            me = bpy.data.meshes.new(f"{name}_{part}")\n            me.from_pydata(verts.tolist(), [], [[remap[a[0] - 1] for a in f] for f in faces]); me.update()\n            uv = me.uv_layers.new(name=\'UVMap\')\n            k = 0\n            for f in faces:\n                for a in f:\n                    t = a[1] - 1; uv.data[k].uv = (VT[t][0], VT[t][1]) if t >= 0 and t < len(VT) else (0, 0); k += 1\n            me.materials.append(mat)\n            for poly in me.polygons: poly.use_smooth = True\n            ob = bpy.data.objects.new(f"{name}_{part}", me); bpy.context.scene.collection.objects.link(ob)\n            ob.rotation_mode = \'QUATERNION\'; self.objs[part] = ob\n            if part in (\'LeftFoot\', \'RightFoot\'): self.foot_pts[part] = P[vi]\n        self.height = P[:, 1].max() - P[:, 1].min()\n        self.V, self.VT, self.G = V, VT, G; self.tex_path = tex_path\n\n    # ---------- Cara 3D: boca que se abre y párpados que parpadean ----------\n    def build_face(self, tex_bgr):\n        ff = rig.face_features(self.V, self.VT, self.G, self.names, self.P, self.J, tex_bgr); self.face = ff\n        head = self.objs[\'Head\']; Jh = self.J[\'Head\']; skin = tuple(ff[\'skin\']) + (1,)\n        def mat(name, col, rough=0.5, emit=0):\n            m = bpy.data.materials.new(name); m.use_nodes = True; b = m.node_tree.nodes[\'Principled BSDF\']\n            b.inputs[\'Base Color\'].default_value = col; b.inputs[\'Roughness\'].default_value = rough\n            try: b.inputs[\'Sheen Weight\'].default_value = 0.3\n            except KeyError: pass\n            return m\n        m_mouth = mat(self.name + \'_boca\', (0.045, 0.004, 0.008, 1), 0.45); m_tongue = mat(self.name + \'_lengua\', (0.85, 0.32, 0.38, 1), 0.4)\n        m_skin = mat(self.name + \'_parpado\', skin, 0.6)\n        c, r = ff[\'mouth\']\n        bpy.ops.mesh.primitive_uv_sphere_add(segments=24, ring_count=12, radius=1); mo = bpy.context.object; mo.name = self.name + \'_boca\'\n        mo.data.materials.append(m_mouth); mo.scale = (r * 1.3, r * 0.18, r * 0.06)\n        bpy.ops.mesh.primitive_uv_sphere_add(segments=16, ring_count=8, radius=1); tg = bpy.context.object; tg.name = self.name + \'_lengua\'\n        tg.data.materials.append(m_tongue); tg.scale = (0.55, 0.6, 0.45); tg.location = (0, -0.35, -0.45); tg.parent = mo\n        mo.location = Vector(to_b(c - Jh)) + Vector((0, r * 0.05, 0)); mo.parent = head\n        for o in (mo, tg):\n            for p in o.data.polygons: p.use_smooth = True\n        self.mouth = mo; self.mouth_r = r\n        self.lids = []\n        for side in (\'L\', \'R\'):\n            ce, re_, rw_ = ff[\'eye_\' + side]\n            bpy.ops.mesh.primitive_uv_sphere_add(segments=24, ring_count=12, radius=1); lid = bpy.context.object; lid.name = f"{self.name}_parpado{side}"\n            lid.data.materials.append(m_skin)\n            for p in lid.data.polygons: p.use_smooth = True\n            # el origen va arriba del ojo: al escalar en Z el párpado "baja"\n            for v in lid.data.vertices: v.co.z -= 1.0\n            lid.scale = (rw_ * 1.08, re_ * 0.3, re_ * 1.02)\n            lid.location = Vector(to_b(ce - Jh)) + Vector((0, -re_ * 0.04, re_ * 1.0)); lid.parent = head\n            self.lids.append(lid)\n        self.set_face(1, 0.0, 0.0)\n\n    def set_face(self, frame, mouth_open, blink, width=0.5):\n        r = self.mouth_r; mo = self.mouth\n        mo.scale = (r * (1.0 + 0.5 * width), r * 0.18, r * (0.04 + 0.85 * mouth_open)); mo.keyframe_insert(\'scale\', frame=frame)\n        for lid in self.lids:\n            s = lid.scale.copy(); lid.scale = (s.x, s.y, max(0.001, (0.02 + 0.98 * blink)) * self._lid_h(lid)); lid.keyframe_insert(\'scale\', frame=frame)\n    def _lid_h(self, lid):\n        if not hasattr(self, \'_lidh\'): self._lidh = {}\n        if lid.name not in self._lidh: self._lidh[lid.name] = lid.scale.z\n        return self._lidh[lid.name]\n\n    # ---------- Pelaje real en cabeza y orejas (se deja libre la cara) ----------\n    def add_fur(self, length=0.028, density=12000, children=8):\n        head = self.objs[\'Head\']; me = head.data; ff = getattr(self, \'face\', None)\n        vg = head.vertex_groups.new(name=\'pelo\'); Jh = self.J[\'Head\']\n        for v in me.vertices:\n            p = np.array(v.co)                                              # local (Blender) relativo a la articulación del cuello\n            w = 1.0\n            if ff is not None:\n                el = to_b(ff[\'eye_L\'][0] - Jh); er = to_b(ff[\'eye_R\'][0] - Jh); mo = to_b(ff[\'mouth\'][0] - Jh)\n                rr = ff[\'eye_L\'][1]\n                # la cara (de las cejas a la barbilla, entre las mejillas) queda sin pelo\n                if (p[1] < min(el[1], er[1]) + rr * 0.6 and mo[2] - rr * 2.2 < p[2] < el[2] + rr * 2.0\n                        and abs(p[0] - (el[0] + er[0]) / 2) < abs(el[0] - er[0]) / 2 + rr * 2.0): w = 0.0\n            vg.add([v.index], w, \'REPLACE\')\n        ps = head.modifiers.new(\'pelaje\', \'PARTICLE_SYSTEM\').particle_system; st = ps.settings\n        st.type = \'HAIR\'; st.count = density; st.hair_length = length * self.height; st.child_type = \'INTERPOLATED\'\n        for k, v in ((\'rendered_child_count\', children), (\'child_percent\', 2), (\'root_radius\', 0.6), (\'tip_radius\', 0.0),\n                     (\'radius_scale\', 0.004 * self.height), (\'use_hair_bspline\', True), (\'display_step\', 3), (\'render_step\', 4),\n                     (\'roughness_2\', 0.15), (\'roughness_endpoint\', 0.05), (\'clump_factor\', 0.2)):\n            try: setattr(st, k, v)\n            except Exception: pass\n        ps.vertex_group_density = \'pelo\'; ps.vertex_group_length = \'pelo\'\n        st.material_slot = head.material_slots[0].name\n\n    def bake_arrays(self, R, Pp, frame0=1):\n        """R (T,15,3,3), Pp (T,15,3) ya en marco y-arriba (orden rig.R15) → keyframes."""\n        for k, part in enumerate(rig.R15):\n            ob = self.objs[part]\n            for t in range(len(Pp)):\n                M = np.eye(4); M[:3, :3] = rot_b(R[t, k]); M[:3, 3] = to_b(Pp[t, k])\n                loc, q, _ = Matrix(M.tolist()).decompose(); ob.location = loc; ob.rotation_quaternion = q\n                ob.keyframe_insert(\'location\', frame=frame0 + t); ob.keyframe_insert(\'rotation_quaternion\', frame=frame0 + t)\n\n    def bake(self, rot, pos, frame0=0, place=np.eye(4)):\n        """rot/pos (marco y-arriba) → keyframes de cada pieza. place = matriz 4x4 Blender (posición + giro en el set)."""\n        T = len(pos[\'LowerTorso\'])\n        Pm = Matrix(place.tolist())\n        for part, ob in self.objs.items():\n            for t in range(T):\n                M = np.eye(4); M[:3, :3] = rot_b(rot[part][t]); M[:3, 3] = to_b(pos[part][t])\n                W = Pm @ Matrix(M.tolist())\n                loc, q, _ = W.decompose(); ob.location = loc; ob.rotation_quaternion = q\n                ob.keyframe_insert(\'location\', frame=frame0 + t); ob.keyframe_insert(\'rotation_quaternion\', frame=frame0 + t)\n\n# =================== Sets, props, textos, cámara ===================\ndef simple_mat(name, col, rough=0.5, metal=0.0, emit=None, strength=1.0, alpha=1.0):\n    m = bpy.data.materials.new(name); m.use_nodes = True; b = m.node_tree.nodes[\'Principled BSDF\']\n    b.inputs[\'Base Color\'].default_value = tuple(col) + (1,) if len(col) == 3 else col\n    b.inputs[\'Roughness\'].default_value = rough; b.inputs[\'Metallic\'].default_value = metal\n    if emit is not None:\n        b.inputs[\'Emission Color\'].default_value = tuple(emit) + (1,); b.inputs[\'Emission Strength\'].default_value = strength\n    if alpha < 1: b.inputs[\'Alpha\'].default_value = alpha\n    return m\ndef key_visible(objs, ranges, total):\n    """Muestra los objetos sólo en los rangos de cuadros [(f0, f1), ...]."""\n    for ob in objs:\n        ob.hide_render = True; ob.keyframe_insert(\'hide_render\', frame=0)\n        for f0, f1 in ranges:\n            ob.hide_render = False; ob.keyframe_insert(\'hide_render\', frame=f0)\n            ob.hide_render = True; ob.keyframe_insert(\'hide_render\', frame=f1)\n        _const(ob)\ndef _const(ob):\n    ad = ob.animation_data\n    if ad is None or ad.action is None: return\n    try: curves = ad.action.fcurves\n    except AttributeError:\n        curves = [c for layer in ad.action.layers for st in layer.strips for cb in st.channelbags for c in cb.fcurves]\n    for fc in curves:\n        if fc.data_path in (\'hide_render\', \'hide_viewport\'):\n            for kp in fc.keyframe_points: kp.interpolation = \'CONSTANT\'\n\ndef build_sets(set_ranges, hdris, total):\n    """set_ranges: {\'blanco\': [(f0,f1)...], \'negro\': [...], \'hdri:<ruta>\': [...]}. Fondo del mundo con mezclas por cuadro."""\n    sc = bpy.context.scene; w = bpy.data.worlds.new(\'mundo\'); sc.world = w; w.use_nodes = True; nt = w.node_tree\n    out = nt.nodes[\'World Output\']; bg_w = nt.nodes[\'Background\']; bg_w.inputs[0].default_value = (1, 1, 1, 1)\n    keys = list(set_ranges); shaders = {}\n    for k in keys:\n        if k == \'blanco\': shaders[k] = (bg_w, 1.0)\n        elif k == \'negro\':\n            n = nt.nodes.new(\'ShaderNodeBackground\'); n.inputs[0].default_value = (0.0, 0.0, 0.0, 1); shaders[k] = (n, 0.0)\n        else:\n            n = nt.nodes.new(\'ShaderNodeBackground\'); env = nt.nodes.new(\'ShaderNodeTexEnvironment\')\n            path = hdris.get(k)\n            if path and os.path.exists(path):\n                env.image = bpy.data.images.load(path); nt.links.new(env.outputs[\'Color\'], n.inputs[\'Color\'])\n            else:\n                sky = nt.nodes.new(\'ShaderNodeTexSky\'); nt.links.new(sky.outputs[\'Color\'], n.inputs[\'Color\']); n.inputs[1].default_value = 0.25\n                shaders[k] = (n, 0.25); continue\n            shaders[k] = (n, 1.0)\n    # cadena de Mix Shader: cada set tiene un factor que vale 1 en sus cuadros\n    cur = shaders[keys[0]][0].outputs[0]; mixes = []\n    for k in keys[1:]:\n        m = nt.nodes.new(\'ShaderNodeMixShader\'); nt.links.new(cur, m.inputs[1]); nt.links.new(shaders[k][0].outputs[0], m.inputs[2]); cur = m.outputs[0]; mixes.append((k, m))\n    nt.links.new(cur, out.inputs[\'Surface\'])\n    for k, m in mixes:\n        fac = m.inputs[0]; fac.default_value = 0; fac.keyframe_insert(\'default_value\', frame=0)\n        for f0, f1 in set_ranges[k]:\n            fac.default_value = 1; fac.keyframe_insert(\'default_value\', frame=f0); fac.default_value = 0; fac.keyframe_insert(\'default_value\', frame=f1)\n    for idb in (w, w.node_tree):\n        if idb.animation_data and idb.animation_data.action: _set_constant(idb)\n    for k, (n, s) in shaders.items():\n        if k == \'blanco\': n.inputs[1].default_value = 0.55\n        elif k != \'negro\': n.inputs[1].default_value = s\n    # pisos y luces de cada set\n    floors = {}\n    bpy.ops.mesh.primitive_plane_add(size=400); fw = bpy.context.object; fw.name = \'piso_blanco\'\n    fw.data.materials.append(simple_mat(\'piso_blanco\', (0.8, 0.8, 0.8), 0.55)); floors[\'blanco\'] = [fw]\n    bpy.ops.mesh.primitive_plane_add(size=400); fn = bpy.context.object; fn.name = \'piso_negro\'\n    fn.data.materials.append(simple_mat(\'piso_negro\', (0.01, 0.01, 0.012), 0.22)); floors[\'negro\'] = [fn]\n    bpy.ops.object.light_add(type=\'SPOT\', location=(0, -6, 16)); sp = bpy.context.object; sp.data.energy = 24000; sp.data.spot_size = math.radians(55)\n    sp.data.spot_blend = 0.6; sp.rotation_euler = (math.radians(20), 0, 0); sp.data.shadow_soft_size = 1.2\n    bpy.ops.object.light_add(type=\'AREA\', location=(-9, 9, 7)); r1 = bpy.context.object; r1.data.energy = 9000; r1.data.color = (0.45, 0.6, 1.0); r1.data.size = 3\n    r1.rotation_euler = (math.radians(-60), 0, math.radians(-135))\n    bpy.ops.object.light_add(type=\'AREA\', location=(9, 9, 7)); r2 = bpy.context.object; r2.data.energy = 9000; r2.data.color = (1.0, 0.55, 0.4); r2.data.size = 3\n    r2.rotation_euler = (math.radians(-60), 0, math.radians(135))\n    floors[\'negro\'] += [sp, r1, r2]\n    bpy.ops.object.light_add(type=\'AREA\', location=(5, -10, 12)); kw = bpy.context.object; kw.data.energy = 3500; kw.data.size = 10\n    kw.rotation_euler = (math.radians(45), 0, math.radians(25)); floors[\'blanco\'].append(kw)\n    for k in keys:\n        if k.startswith(\'hdri\'):\n            bpy.ops.mesh.primitive_plane_add(size=400); fc = bpy.context.object; fc.name = \'piso_sombras_\' + k[5:]\n            fc.is_shadow_catcher = True; fc.data.materials.append(simple_mat(\'sombra\', (0.5, 0.5, 0.5))); floors[k] = [fc]\n    for k, objs in floors.items():\n        if k in set_ranges: key_visible(objs, set_ranges[k], total)\n        else: key_visible(objs, [], total)\n        for ob in objs: _const(ob)\ndef _set_constant(idb):\n    try:\n        for fc in idb.animation_data.action.fcurves:\n            for kp in fc.keyframe_points: kp.interpolation = \'CONSTANT\'\n    except AttributeError:\n        for layer in idb.animation_data.action.layers:\n            for st in layer.strips:\n                for cb in st.channelbags:\n                    for fc in cb.fcurves:\n                        for kp in fc.keyframe_points: kp.interpolation = \'CONSTANT\'\n    except Exception: pass\n\ndef text3d(txt, font_path, size, mat, loc, rot_z=0.0, extrude=0.12):\n    cu = bpy.data.curves.new(\'t\', \'FONT\'); cu.body = txt; cu.align_x = \'CENTER\'; cu.align_y = \'CENTER\'; cu.size = size\n    cu.extrude = extrude; cu.bevel_depth = 0.02; cu.bevel_resolution = 2\n    if font_path and os.path.exists(font_path):\n        try: cu.font = bpy.data.fonts.load(font_path)\n        except Exception: pass\n    ob = bpy.data.objects.new(\'texto\', cu); bpy.context.scene.collection.objects.link(ob)\n    ob.data.materials.append(mat); ob.location = loc; ob.rotation_euler = (math.radians(90), 0, rot_z); return ob\ndef animate_pop(ob, f0, f1, base_scale=1.0, rise=0.6):\n    """Aparece creciendo con rebote y subiendo un poco; desaparece al final."""\n    z = ob.location.z\n    ob.scale = (0.001,) * 3; ob.location.z = z - rise; ob.keyframe_insert(\'scale\', frame=f0); ob.keyframe_insert(\'location\', frame=f0)\n    ob.scale = (base_scale * 1.12,) * 3; ob.location.z = z + 0.05; ob.keyframe_insert(\'scale\', frame=f0 + 6); ob.keyframe_insert(\'location\', frame=f0 + 6)\n    ob.scale = (base_scale,) * 3; ob.location.z = z; ob.keyframe_insert(\'scale\', frame=f0 + 10); ob.keyframe_insert(\'location\', frame=f0 + 10)\n    ob.keyframe_insert(\'scale\', frame=max(f0 + 11, f1 - 6)); ob.scale = (0.001,) * 3; ob.keyframe_insert(\'scale\', frame=f1)\n\ndef make_chair(height, mat):\n    """Silla estilo Roblox (bloques) con el asiento a la altura indicada."""\n    parts = []\n    def box(sx, sy, sz, x, y, z):\n        bpy.ops.mesh.primitive_cube_add(size=1, location=(x, y, z)); b = bpy.context.object; b.scale = (sx, sy, sz)\n        bev = b.modifiers.new(\'b\', \'BEVEL\'); bev.width = 0.04; bev.segments = 2; b.data.materials.append(mat); parts.append(b)\n    W = 2.2; th = 0.22\n    box(W, W, th, 0, 0, height - th / 2)\n    for x in (-W / 2 + 0.15, W / 2 - 0.15):\n        for y in (-W / 2 + 0.15, W / 2 - 0.15): box(0.22, 0.22, height - th, x, y, (height - th) / 2)\n    box(W, 0.22, 2.4, 0, W / 2 - 0.11, height + 1.2)\n    bpy.ops.object.empty_add(location=(0, 0, 0)); root = bpy.context.object; root.name = \'silla\'\n    for p_ in parts: p_.parent = root\n    return root, parts\ndef make_gizmo(size):\n    """Flechas de mover de Roblox Studio (rojo X, verde Y, azul Z) + caja de selección."""\n    bpy.ops.object.empty_add(); root = bpy.context.object; root.name = \'gizmo\'; objs = []\n    for axis, col in (((1, 0, 0), (1, 0.1, 0.1)), ((0, 1, 0), (0.1, 1, 0.2)), ((0, 0, 1), (0.15, 0.4, 1))):\n        m = simple_mat(\'flecha\', col, 0.4, emit=col, strength=2.5)\n        for sgn in (1, -1):\n            d = Vector(axis) * sgn\n            bpy.ops.mesh.primitive_cylinder_add(radius=0.06 * size, depth=size, location=d * size * 0.75)\n            c = bpy.context.object; c.rotation_euler = d.to_track_quat(\'Z\', \'Y\').to_euler(); c.data.materials.append(m); c.parent = root; objs.append(c)\n            bpy.ops.mesh.primitive_cone_add(radius1=0.18 * size, depth=0.4 * size, location=d * size * 1.4)\n            k = bpy.context.object; k.rotation_euler = d.to_track_quat(\'Z\', \'Y\').to_euler(); k.data.materials.append(m); k.parent = root; objs.append(k)\n    bpy.ops.mesh.primitive_cube_add(size=1); box = bpy.context.object; box.scale = (size * 0.9, size * 0.7, size * 1.6)\n    wf = box.modifiers.new(\'w\', \'WIREFRAME\'); wf.thickness = 0.04; box.data.materials.append(simple_mat(\'sel\', (0.1, 0.6, 1), 0.3, emit=(0.1, 0.6, 1), strength=4))\n    box.parent = root; objs.append(box)\n    return root, objs\ndef billboard_image(img_path, width, loc, cam):\n    img = bpy.data.images.load(img_path); asp = img.size[1] / max(1, img.size[0])\n    bpy.ops.mesh.primitive_plane_add(size=1, location=loc); pl = bpy.context.object; pl.scale = (width, width * asp, 1)\n    m = bpy.data.materials.new(\'bb\'); m.use_nodes = True; nt = m.node_tree; b = nt.nodes[\'Principled BSDF\']\n    t = nt.nodes.new(\'ShaderNodeTexImage\'); t.image = img; nt.links.new(t.outputs[\'Color\'], b.inputs[\'Base Color\'])\n    nt.links.new(t.outputs[\'Color\'], b.inputs[\'Emission Color\']); b.inputs[\'Emission Strength\'].default_value = 0.8; nt.links.new(t.outputs[\'Alpha\'], b.inputs[\'Alpha\'])\n    pl.data.materials.append(m)\n    c = pl.constraints.new(\'TRACK_TO\'); c.target = cam; c.track_axis = \'TRACK_Z\'; c.up_axis = \'UP_Y\'\n    return pl\n', 'promo': '# Pipeline de la promo: guion → tiempos → movimiento → retarget → cara → cámara → spec para Blender.\nimport numpy as np, math, json, os\nimport rig, movimiento as mv, director as dr, camara, escena_math as em\nFPS = mv.FPS\n\ndef default_beats(server="The Republic of Furrys", dominio="therepublicoffury.web.app"):\n    idle = {\'prompt\': "A person stands relaxed, breathing and shifting weight slightly.", \'kind\': \'idle\'}\n    sit_idle = {\'prompt\': "A person sits on a chair, listening and nodding slightly.", \'kind\': \'sit_idle\'}\n    phone = {\'prompt\': "A person stands holding a phone in the right hand in front of the chest, looking at the screen.", \'kind\': \'phone\'}\n    return [\n        {\'id\': \'gancho\', \'set\': \'blanco\', \'cam\': \'medio_A\', \'habla\': \'A\', \'linea\': "¡Jajaja, mira esto! Otro server de Discord… todos están muertos, no sirven.",\n         \'A\': {\'prompt\': "A person stands holding a phone in the right hand, looks at the screen and laughs, shaking the shoulders.", \'kind\': \'phone_laugh\'},\n         \'B\': {\'prompt\': "A person stands with arms crossed, watching and tilting the head.", \'kind\': \'idle\'}},\n        {\'id\': \'no\', \'set\': \'blanco\', \'cam\': \'cerca_B\', \'habla\': \'B\', \'linea\': "No, no, no… a ver. Siéntate.",\n         \'A\': phone, \'B\': {\'prompt\': "A person shakes the head and raises the right hand with the palm forward.", \'kind\': \'talk_no\'}},\n        {\'id\': \'comando\', \'set\': \'blanco\', \'cam\': \'amplio\', \'habla\': None, \'dur\': 2.0, \'evento\': \'silla\',\n         \'A\': phone, \'B\': {\'prompt\': "A person raises the right arm quickly and snaps the fingers.", \'kind\': \'snap\'}},\n        {\'id\': \'arrastre\', \'set\': \'blanco\', \'cam\': \'amplio\', \'habla\': \'A\', \'linea\': "¿Eh? ¡Oye, bájame!", \'evento\': \'flechas\',\n         \'A\': {\'prompt\': "A person flails both arms and kicks the legs in panic.", \'kind\': \'flail\', \'drag\': True},\n         \'B\': {\'prompt\': "A person extends the right arm forward and slowly moves the hand to the left as if dragging something.", \'kind\': \'drag_hand\'}},\n        {\'id\': \'sienta\', \'set\': \'blanco\', \'cam\': \'medio_A\', \'habla\': None, \'dur\': 1.6,\n         \'A\': {\'prompt\': "A person sits down on a chair.", \'kind\': \'sit\'}, \'B\': idle},\n        {\'id\': \'republica\', \'set\': \'negro\', \'cam\': \'cerca_B\', \'habla\': \'B\', \'linea\': "Esto no es un server. Es una república.",\n         \'A\': sit_idle, \'B\': {\'prompt\': "A person talks calmly while opening both hands to the sides.", \'kind\': \'talk\'},\n         \'textos\': [server.upper()]},\n        {\'id\': \'detalles\', \'set\': \'hdri:plaza\', \'cam\': \'medio_B\', \'habla\': \'B\', \'linea\': "Tenemos presidente, elecciones, partidos políticos… y hasta economía.",\n         \'A\': sit_idle, \'B\': {\'prompt\': "A person explains something, counting on the fingers and gesturing with both hands.", \'kind\': \'talk_count\'},\n         \'textos\': ["PRESIDENTE", "ELECCIONES", "PARTIDOS", "ECONOMÍA"]},\n        {\'id\': \'pregunta\', \'set\': \'hdri:plaza\', \'cam\': \'pov_B\', \'habla\': \'A\', \'linea\': "¿Y yo qué haría ahí?",\n         \'A\': {\'prompt\': "A person sitting on a chair shrugs with both hands raised.", \'kind\': \'sit_shrug\'}, \'B\': idle},\n        {\'id\': \'respuesta\', \'set\': \'hdri:bosque\', \'cam\': \'pov_A\', \'habla\': \'B\', \'linea\': "Votas, te lanzas a diputado… o fundas tu propio partido.",\n         \'A\': sit_idle, \'B\': {\'prompt\': "A person talks enthusiastically, pointing forward and then opening both arms.", \'kind\': \'talk\'},\n         \'textos\': ["VOTA", "GOBIERNA", "FUNDA TU PARTIDO"]},\n        {\'id\': \'levanta\', \'set\': \'negro\', \'cam\': \'medio_A\', \'habla\': \'A\', \'linea\': "…¿Dónde me uno?",\n         \'A\': {\'prompt\': "A person stands up quickly from a chair with excitement.", \'kind\': \'stand_up\'}, \'B\': idle},\n        {\'id\': \'cta\', \'set\': \'blanco\', \'cam\': \'cerca_B\', \'habla\': \'B\', \'linea\': "Link en el perfil.",\n         \'A\': idle, \'B\': {\'prompt\': "A person points forward at the camera with the right hand and nods.", \'kind\': \'point\', \'mira_camara\': True},\n         \'textos\': ["LINK EN EL PERFIL", dominio]},\n    ]\n\ndef avatar_data(obj, tex_bgr):\n    V, VT, G, _ = rig.load_obj(obj); names, info, P = rig.classify(V, G); J = rig.joints_r15(names, info, P)\n    by = {n: g for g, n in names.items()}\n    pts = {p: P[info[by[p]][\'vi\']] for p in rig.R15}\n    face = rig.face_features(V, VT, G, names, P, J, tex_bgr)\n    return {\'J\': J, \'pts\': pts, \'face\': face, \'height\': P[:, 1].max() - P[:, 1].min()}\n\ndef retarget_full(clip, smpl_rest, ad):\n    rot, pos, scale = rig.retarget(clip.poses, clip.trans, smpl_rest, ad[\'J\'], {f: ad[\'pts\'][f] for f in (\'LeftFoot\', \'RightFoot\')}, absolute=True)\n    R = np.stack([rot[p] for p in rig.R15], 1); P = np.stack([pos[p] for p in rig.R15], 1)\n    return R, P, scale\n\ndef scale_of(smpl_rest, ad):\n    idx = {n: i for i, n in enumerate(rig.SMPL_NAMES)}; J = ad[\'J\']\n    smpl_hip = smpl_rest[idx[\'Pelvis\'], 1] - min(smpl_rest[idx[\'L_Ankle\'], 1], smpl_rest[idx[\'R_Ankle\'], 1])\n    return (J[\'LowerTorso\'][1] - min(J[\'LeftFoot\'][1], J[\'RightFoot\'][1])) / smpl_hip\n\ndef assemble(beats, voices, get_clip, avatars, smpl_rest, font=None, hdris=None):\n    """voices = {beat_id: (wav_float32, sr)}; avatars = {\'A\': avatar_data, \'B\': ...} → (arrays npz, spec parcial)."""\n    total = dr.plan(beats, {k: len(w) / sr for k, (w, sr) in voices.items()})\n    out, infos = {}, {}\n    for who in (\'A\', \'B\'):\n        sc = scale_of(smpl_rest, avatars[who])\n        clip, info = dr.build_character(who, beats, get_clip, sc, total)\n        R, P, _ = retarget_full(clip, smpl_rest, avatars[who]); infos[who] = info\n        out[\'R\' + who], out[\'P\' + who] = R, P\n    # mirada: quien escucha sigue con la vista a quien habla\n    heads = {}; fwds = {}\n    for who in (\'A\', \'B\'):\n        heads[who], fwds[who] = camara.head_points(out[\'R\' + who], out[\'P\' + who], avatars[who][\'J\'], avatars[who][\'face\'])\n    for who in (\'A\', \'B\'):\n        other = \'B\' if who == \'A\' else \'A\'; w = np.zeros(total)\n        for b in beats:\n            if b.get(\'habla\') == other or b.get(\'habla\') is None: w[b[\'f0\']:b[\'f0\'] + b[\'n\']] = 0.85\n            if b[who].get(\'kind\') in (\'phone\', \'phone_laugh\') and b.get(\'habla\') != other: w[b[\'f0\']:b[\'f0\'] + b[\'n\']] = 0.0\n        w = np.convolve(w, np.ones(9) / 9, \'same\')\n        rot = {p: out[\'R\' + who][:, k] for k, p in enumerate(rig.R15)}; pos = {p: out[\'P\' + who][:, k] for k, p in enumerate(rig.R15)}\n        oh = em.to_b(np.zeros(3))\n        tgt_y = (out[\'P\' + other][:, 2] + np.array([0, avatars[other][\'height\'] * 0.12, 0]))\n        rot = mv.look_at(rot, pos, avatars[who][\'J\'], lambda t: tgt_y[t], w)\n        out[\'R\' + who] = np.stack([rot[p] for p in rig.R15], 1)\n        heads[who], fwds[who] = camara.head_points(out[\'R\' + who], out[\'P\' + who], avatars[who][\'J\'], avatars[who][\'face\'])\n        out[\'H\' + who] = heads[who]\n    # cara: labios con la voz de cada quien + parpadeos\n    for who in (\'A\', \'B\'):\n        mo = np.zeros(total); wi = np.full(total, 0.5)\n        for b in beats:\n            if b.get(\'habla\') == who and b[\'id\'] in voices:\n                wav, sr = voices[b[\'id\']]; o, w_ = mv.lipsync(wav, sr, total, b[\'voz_f\'])\n                mo = np.maximum(mo, o); m_ = o > 0; wi[m_] = w_[m_]\n        out[\'mouth\' + who] = mo; out[\'width\' + who] = wi; out[\'blink\' + who] = mv.blinks(total, 7 if who == \'A\' else 11)\n    loc, tgt, lens = camara.solve(beats, heads, fwds, total)\n    out[\'cam_loc\'], out[\'cam_tgt\'], out[\'cam_lens\'] = loc, tgt, lens\n    # sets por rangos de cuadros (Blender empieza en el cuadro 1)\n    sets = {}\n    for b in beats: sets.setdefault(b[\'set\'], []).append([b[\'f0\'] + 1, b[\'f0\'] + b[\'n\'] + 1])\n    spec = {\'total\': total, \'sets\': sets, \'cuts\': [b[\'f0\'] + 1 for b in beats]}\n    # silla: altura del asiento = punto más bajo de la cadera/muslos de A ya sentado\n    sit = infos[\'A\'].get(\'sit\')\n    if sit:\n        f = min(total - 1, sit[\'f1\'] - 1); RA, PA = out[\'RA\'], out[\'PA\']; J = avatars[\'A\'][\'J\']; low = 1e9\n        for p in (\'LowerTorso\', \'LeftUpperLeg\', \'RightUpperLeg\'):\n            k = rig.R15.index(p); v = np.einsum(\'ij,nj->ni\', RA[f, k], avatars[\'A\'][\'pts\'][p] - J[p]) + PA[f, k]; low = min(low, v[:, 1].min())\n        fwd = em.to_b(RA[f, 0] @ np.array([0, 0, 1.0])); xy = em.to_b(PA[f, 0])[:2] - fwd[:2] / (np.linalg.norm(fwd[:2]) + 1e-9) * 0.25\n        f_spawn = next(b[\'f0\'] for b in beats if b.get(\'evento\') == \'silla\') + int(0.75 * FPS)\n        spec[\'chair\'] = {\'height\': float(max(0.6, low)), \'xy\': [float(xy[0]), float(xy[1])], \'yaw\': float(math.atan2(fwd[0], -fwd[1])), \'f_spawn\': f_spawn + 1}\n    drag = infos[\'A\'].get(\'drag\')\n    if drag: spec[\'gizmo\'] = {\'who\': \'A\', \'f0\': drag[\'f0\'] + 1, \'f1\': drag[\'f1\'] + 1}\n    cmd = next((b for b in beats if b.get(\'evento\') == \'silla\'), None)\n    if cmd: spec[\'bubble_range\'] = [cmd[\'f0\'], cmd[\'f0\'] + cmd[\'n\']]\n    ph = [[b[\'f0\'] + 1, b[\'f0\'] + b[\'n\'] + 1] for b in beats if b[\'A\'].get(\'kind\') in (\'phone\', \'phone_laugh\')]\n    spec[\'phone_ranges\'] = ph\n    # textos 3D: detrás de los personajes (siguiendo la línea de la cámara), de frente a ella y en perspectiva\n    texts = []\n    for b in beats:\n        items = b.get(\'textos\') or []\n        if not items: continue\n        style = \'hdri\' if b[\'set\'].startswith(\'hdri\') else b[\'set\']\n        f = b[\'f0\'] + b[\'n\'] // 2; L, Tg = loc[f], tgt[f]\n        v = Tg - L; v[2] = 0; dist = np.linalg.norm(v); v /= (dist + 1e-9); side = np.array([v[1], -v[0], 0])\n        D = dist + 5.0 + 0.25 * dist; back = Tg + v * (D - dist)        # detrás del sujeto\n        tan_v = 12.0 / lens[f]; Wv = 2 * tan_v * (9 / 16) * D; Hv = 2 * tan_v * D   # tamaño visible del cuadro (vertical) a esa distancia\n        n = len(items); mx = max(len(t) for t in items)\n        size = float(min(2.4, 0.85 * Wv / (0.62 * max(6, mx))))\n        for i, txt in enumerate(items):                                    # en columna, arriba de la cabeza, como letrero gigante\n            p = back + np.array([0, 0, 0.18 * Hv - i * 1.35 * size])\n            f0 = b[\'f0\'] + int(0.25 * FPS) + int(i * b[\'n\'] / max(1, n) * 0.55)\n            texts.append({\'txt\': txt, \'f0\': f0, \'f1\': b[\'f0\'] + b[\'n\'], \'style\': style, \'loc\': [float(c) for c in p], \'size\': size,\n                          \'rot\': float(math.atan2(-v[0], v[1]))})\n    spec[\'texts\'] = texts\n    return out, spec, infos\n', 'audio_promo': '# Sonido de la promo: voces + efectos (silla, flechas, cortes) + música generada con ritmo, con ducking bajo las voces.\nimport numpy as np\nSR = 48000; FPS = 30\nrng = np.random.default_rng(5)\ndef _lp(x, a=0.15):\n    y = np.empty_like(x); acc = 0.0\n    for i in range(len(x)): acc += a * (x[i] - acc); y[i] = acc\n    return y\ndef noise(d): return rng.normal(0, 1, int(d * SR)).astype(np.float32)\ndef env(n, a, r):\n    e = np.ones(n, np.float32); na, nr = min(n, max(1, int(a * SR))), min(n, max(1, int(r * SR))); e[:na] = np.linspace(0, 1, na); e[-nr:] *= np.linspace(1, 0, nr) ** 2; return e\ndef whoosh(d=0.5):\n    n = noise(d); t = np.linspace(0, 1, len(n)); hp = n - np.convolve(n, np.ones(40) / 40, \'same\'); return (hp * np.sin(np.pi * t) ** 2 * 0.25).astype(np.float32)\ndef pop():\n    t = np.arange(int(0.25 * SR)) / SR; return (np.sin(2 * np.pi * (300 + 900 * t) * t) * np.exp(-t * 22) * 0.6).astype(np.float32)\ndef sparkle(d=0.9):\n    t = np.arange(int(d * SR)) / SR; out = np.zeros_like(t)\n    for k in range(14):\n        st = rng.uniform(0, d * 0.7); f = rng.uniform(2500, 6000); m = t >= st\n        out[m] += np.sin(2 * np.pi * f * (t[m] - st)) * np.exp(-(t[m] - st) * 18) * 0.12\n    return out.astype(np.float32)\ndef click():\n    t = np.arange(int(0.06 * SR)) / SR; return (np.sin(2 * np.pi * 1800 * t) * np.exp(-t * 90) * 0.5).astype(np.float32)\ndef thump():\n    t = np.arange(int(0.4 * SR)) / SR; return (np.sin(2 * np.pi * (90 - 40 * t) * t) * np.exp(-t * 12) * 0.8).astype(np.float32)\ndef music(L, bpm=104):\n    """Base pegajosa: bombo, hi-hat, bajo y acordes (Do mayor → La menor → Fa → Sol)."""\n    n = int(L * SR); out = np.zeros(n, np.float32); beat = 60 / bpm; t = np.arange(n) / SR\n    chords = [[60, 64, 67], [57, 60, 64], [53, 57, 60], [55, 59, 62]]\n    hz = lambda m: 440 * 2 ** ((m - 69) / 12)\n    k = 0\n    while k * beat < L:\n        st = int(k * beat * SR)\n        kick = thump()[:int(0.3 * SR)] * (1.0 if k % 2 == 0 else 0.0)\n        hat = (np.diff(noise(0.05), prepend=0) * np.exp(-np.arange(int(0.05 * SR)) / SR * 80) * 0.08).astype(np.float32)\n        for snd, off in ((kick, 0), (hat, int(beat * SR / 2))):\n            a = st + off; b = min(n, a + len(snd))\n            if b > a: out[a:b] += snd[:b - a]\n        ch = chords[(k // 4) % 4]\n        if k % 4 == 0 and n - st > 100:\n            d = int(beat * 4 * SR); b = min(n, st + d); tt = np.arange(b - st) / SR\n            pad = sum(np.sin(2 * np.pi * hz(m) * tt) + 0.3 * np.sin(4 * np.pi * hz(m) * tt) for m in ch) * 0.025 * env(b - st, 0.05, 0.3)\n            out[st:b] += pad\n        bd = int(beat * 0.9 * SR); b = min(n, st + bd); tt = np.arange(max(0, b - st)) / SR\n        if b > st: out[st:b] += np.sin(2 * np.pi * hz(ch[0] - 24) * tt) * np.exp(-tt * 3) * 0.18\n        k += 1\n    return out\ndef mix(beats, voices, total, spec, vol_music=0.5):\n    L = total / FPS + 0.5; n = int(L * SR); voz = np.zeros(n, np.float32); fx = np.zeros(n, np.float32)\n    def add(buf, x, t, g=1.0):\n        a = int(t * SR); b = min(n, a + len(x))\n        if 0 <= a < b: buf[a:b] += x[:b - a] * g\n    for b in beats:\n        if b[\'id\'] in voices:\n            w, sr = voices[b[\'id\']]\n            if sr != SR: w = np.interp(np.arange(int(len(w) * SR / sr)) * sr / SR, np.arange(len(w)), w).astype(np.float32)\n            add(voz, w, b[\'voz_f\'] / FPS)\n        if b[\'f0\'] > 0: add(fx, whoosh(0.35), b[\'f0\'] / FPS - 0.18, 0.5)\n    ch = spec.get(\'chair\')\n    if ch: add(fx, pop(), (ch[\'f_spawn\'] - 1) / FPS); add(fx, sparkle(), (ch[\'f_spawn\'] - 1) / FPS, 0.8)\n    gz = spec.get(\'gizmo\')\n    if gz: add(fx, click(), (gz[\'f0\'] - 13) / FPS); add(fx, whoosh(1.2), (gz[\'f0\'] - 1) / FPS, 0.6)\n    bb = spec.get(\'bubble\')\n    if bb: add(fx, click(), bb[\'f0\'] / FPS + 0.1, 0.7)\n    for t_ in spec.get(\'texts\', []): add(fx, whoosh(0.25), t_[\'f0\'] / FPS, 0.25)\n    mus = music(L)[:n]\n    env_v = np.sqrt(np.convolve(voz ** 2, np.ones(2400) / 2400, \'same\')); duck = 1 - 0.6 * np.clip(env_v * 10, 0, 1)\n    duck = np.convolve(duck, np.ones(SR // 4) / (SR // 4), \'same\')\n    m = voz * 1.0 + mus * duck * vol_music + fx * 0.8\n    m /= max(1.0, np.abs(m).max() / 0.95)\n    return np.stack([m, m], 1)\n', 'job': '# Se ejecuta con Python + bpy (proceso aparte): arma la escena completa y renderiza los cuadros pedidos.\nimport sys, os, json, math, time, numpy as np\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport bpy, cv2, rig, escena\nfrom mathutils import Vector, Matrix, Euler\nspec = json.load(open(sys.argv[1])); D = np.load(spec[\'npz\'])\nT = int(spec[\'total\'])\nescena.clear_scene(); sc = bpy.context.scene; sc.frame_start = 1; sc.frame_end = T; sc.render.fps = spec[\'render\'][\'fps\']\nt0 = time.time()\n# ---------- personajes ----------\nAV = {}\nfor who, a in spec[\'avatars\'].items():\n    av = escena.Avatar(a[\'name\'], a[\'obj\'], a[\'tex\']); av.bake_arrays(D[\'R\' + who], D[\'P\' + who], 1)\n    av.build_face(cv2.imread(a[\'tex\'], cv2.IMREAD_UNCHANGED))\n    if a.get(\'fur\', True): av.add_fur(length=a.get(\'fur_len\', 0.028), density=a.get(\'fur_density\', 12000))\n    mo, wi, bl = D[\'mouth\' + who], D[\'width\' + who], D[\'blink\' + who]\n    for f in range(T): av.set_face(f + 1, float(mo[f]), float(bl[f]), float(wi[f]))\n    AV[who] = av\nprint(f"personajes listos ({time.time() - t0:.0f}s)", flush=True)\n# ---------- sets ----------\nescena.build_sets({k: [tuple(r) for r in v] for k, v in spec[\'sets\'].items()}, spec.get(\'hdris\', {}), T)\n# ---------- cámara (una sola, animada cuadro por cuadro) ----------\nbpy.ops.object.empty_add(); tgt = bpy.context.object; tgt.name = \'objetivo\'\nbpy.ops.object.camera_add(); cam = bpy.context.object; sc.camera = cam\nc = cam.constraints.new(\'TRACK_TO\'); c.target = tgt; c.track_axis = \'TRACK_NEGATIVE_Z\'; c.up_axis = \'UP_Y\'\ncam.data.sensor_fit = \'VERTICAL\' if spec[\'render\'][\'h\'] > spec[\'render\'][\'w\'] else \'HORIZONTAL\'; cam.data.sensor_height = 24; cam.data.sensor_width = 36\ncl, ct, cn = D[\'cam_loc\'], D[\'cam_tgt\'], D[\'cam_lens\']\ncuts = set(spec.get(\'cuts\', []))\nfor f in range(T):\n    cam.location = Vector(cl[f]); tgt.location = Vector(ct[f]); cam.data.lens = float(cn[f])\n    cam.keyframe_insert(\'location\', frame=f + 1); tgt.keyframe_insert(\'location\', frame=f + 1); cam.data.keyframe_insert(\'lens\', frame=f + 1)\ndef _curves(idb):\n    ad = idb.animation_data\n    if ad is None or ad.action is None: return []\n    try: return list(ad.action.fcurves)\n    except AttributeError: return [c for l in ad.action.layers for s_ in l.strips for cb in s_.channelbags for c in cb.fcurves]\nfor idb in (cam, tgt, cam.data):                       # en los cortes la cámara salta sin barrido (sin desenfoque de movimiento falso)\n    for fc in _curves(idb):\n        for kp in fc.keyframe_points:\n            if int(round(kp.co[0])) + 1 in cuts: kp.interpolation = \'CONSTANT\'\nif spec[\'render\'].get(\'dof\', True):\n    cam.data.dof.use_dof = True; cam.data.dof.focus_object = tgt; cam.data.dof.aperture_fstop = spec[\'render\'].get(\'fstop\', 4.0)\n# ---------- utilería ----------\nph = spec.get(\'phone\')\nif ph:\n    av = AV[ph[\'who\']]; hand = av.objs[\'RightHand\']; Hs = av.height\n    bpy.ops.mesh.primitive_cube_add(size=1); phone = bpy.context.object; phone.name = \'celular\'\n    phone.scale = (0.32 * Hs / 6, 0.035 * Hs / 6, 0.62 * Hs / 6)\n    bev = phone.modifiers.new(\'b\', \'BEVEL\'); bev.width = 0.05; bev.segments = 4\n    phone.data.materials.append(escena.simple_mat(\'carcasa\', (0.02, 0.02, 0.025), 0.3, 0.6))\n    bpy.ops.mesh.primitive_plane_add(size=1); scr = bpy.context.object; scr.name = \'pantalla\'\n    scr.scale = (0.29 * Hs / 6, 0.58 * Hs / 6, 1); scr.rotation_euler = (math.radians(90), 0, 0); scr.location = (0, 0.02 * Hs / 6 + 0.001, 0)\n    m = bpy.data.materials.new(\'pantalla\'); m.use_nodes = True; nt = m.node_tree; b = nt.nodes[\'Principled BSDF\']\n    tx = nt.nodes.new(\'ShaderNodeTexImage\'); tx.image = bpy.data.images.load(ph[\'screen\'])\n    nt.links.new(tx.outputs[\'Color\'], b.inputs[\'Emission Color\']); b.inputs[\'Emission Strength\'].default_value = 2.0\n    b.inputs[\'Base Color\'].default_value = (0, 0, 0, 1); scr.data.materials.append(m)\n    scr.parent = phone; scr.location = (0, 0.6, 0); scr.scale = (0.9, 0.9, 1); scr.rotation_euler = (math.radians(90), 0, 0)\n    tip = np.array(ph[\'tip_local\'])\n    phone.parent = hand; phone.location = Vector(tip); phone.rotation_euler = Euler(ph.get(\'rot\', (math.radians(90), 0, 0)))\n    escena.key_visible([phone, scr], [tuple(r) for r in ph[\'ranges\']], T)\nch = spec.get(\'chair\')\nif ch:\n    root, parts = escena.make_chair(ch[\'height\'], escena.simple_mat(\'madera\', (0.42, 0.24, 0.12), 0.45))\n    root.location = (ch[\'xy\'][0], ch[\'xy\'][1], 0); root.rotation_euler = (0, 0, ch[\'yaw\'])\n    f = ch[\'f_spawn\']\n    root.scale = (0.001,) * 3; root.keyframe_insert(\'scale\', frame=f); root.scale = (1.15,) * 3; root.keyframe_insert(\'scale\', frame=f + 6)\n    root.scale = (0.95,) * 3; root.keyframe_insert(\'scale\', frame=f + 10); root.scale = (1,) * 3; root.keyframe_insert(\'scale\', frame=f + 14)\n    escena.key_visible(parts, [(f, T + 1)], T)\n    # chispas al aparecer\n    sp_m = escena.simple_mat(\'chispa\', (1, 0.9, 0.5), 0.3, emit=(1, 0.85, 0.4), strength=12)\n    rng = np.random.default_rng(1)\n    for k in range(26):\n        bpy.ops.mesh.primitive_ico_sphere_add(subdivisions=1, radius=0.07); s_ = bpy.context.object; s_.data.materials.append(sp_m)\n        d = rng.normal(size=3); d[2] = abs(d[2]) + 0.5; d /= np.linalg.norm(d)\n        p0 = Vector((ch[\'xy\'][0], ch[\'xy\'][1], ch[\'height\'] * 0.7)); s_.location = p0; s_.keyframe_insert(\'location\', frame=f)\n        s_.location = p0 + Vector(d * rng.uniform(2, 4)); s_.keyframe_insert(\'location\', frame=f + 14)\n        s_.scale = (1,) * 3; s_.keyframe_insert(\'scale\', frame=f + 2); s_.scale = (0.01,) * 3; s_.keyframe_insert(\'scale\', frame=f + 16)\n        escena.key_visible([s_], [(f, f + 17)], T)\ngz = spec.get(\'gizmo\')\nif gz:\n    av = AV[gz[\'who\']]; root, objs = escena.make_gizmo(av.height * 0.32)\n    Pp = D[\'P\' + gz[\'who\']]\n    for f in range(max(0, gz[\'f0\'] - 12), min(T, gz[\'f1\'] + 4)):\n        root.location = Vector(escena.to_b(Pp[f, 0]) + np.array([0, 0, av.height * 0.05])); root.keyframe_insert(\'location\', frame=f + 1)\n    escena.key_visible(objs, [(max(1, gz[\'f0\'] - 12), gz[\'f1\'] + 4)], T)\nbb = spec.get(\'bubble\')\nif bb:\n    pl = escena.billboard_image(bb[\'img\'], AV[bb[\'who\']].height * 0.42, (0, 0, 0), cam)\n    H = D[\'H\' + bb[\'who\']]\n    for f in range(bb[\'f0\'], min(T, bb[\'f1\'])):\n        pl.location = Vector(H[f] + np.array([0, 0, AV[bb[\'who\']].height * 0.33])); pl.keyframe_insert(\'location\', frame=f + 1)\n    escena.key_visible([pl], [(bb[\'f0\'] + 1, bb[\'f1\'] + 1)], T)\n# ---------- textos 3D en perspectiva detrás de ellos ----------\nmats = {\'blanco\': escena.simple_mat(\'t_blanco\', (0.02, 0.02, 0.025), 0.25, 0.3),\n        \'negro\': escena.simple_mat(\'t_negro\', (1, 0.85, 0.5), 0.3, 0.8, emit=(1, 0.8, 0.45), strength=5),\n        \'hdri\': escena.simple_mat(\'t_hdri\', (1, 0.78, 0.25), 0.3, 0.9, emit=(1, 0.7, 0.2), strength=2.5)}\nfor t_ in spec.get(\'texts\', []):\n    ob = escena.text3d(t_[\'txt\'], spec.get(\'font\'), t_[\'size\'], mats[t_[\'style\']], t_[\'loc\'], t_.get(\'rot\', 0))\n    escena.animate_pop(ob, t_[\'f0\'] + 1, t_[\'f1\'] + 1)\n    escena.key_visible([ob], [(t_[\'f0\'] + 1, t_[\'f1\'] + 1)], T)\n# ---------- render ----------\nr = spec[\'render\']; sc.render.engine = \'CYCLES\'\nsc.render.resolution_x, sc.render.resolution_y = r[\'w\'], r[\'h\']; sc.render.resolution_percentage = 100\ncp = bpy.context.preferences.addons[\'cycles\'].preferences; dev = \'CPU\'\nfor kind in (\'OPTIX\', \'CUDA\'):\n    try:\n        cp.compute_device_type = kind; cp.get_devices()\n        if any(d.type == kind for d in cp.devices):\n            for d in cp.devices: d.use = d.type == kind\n            dev = \'GPU\'; break\n    except Exception: pass\nsc.cycles.device = dev; print("Render en", dev, cp.compute_device_type, flush=True)\nsc.cycles.samples = r[\'samples\']; sc.cycles.use_adaptive_sampling = True; sc.cycles.adaptive_threshold = 0.02\nsc.cycles.use_denoising = True\ntry: sc.cycles.denoiser = \'OPENIMAGEDENOISE\'; sc.cycles.denoising_use_gpu = True\nexcept Exception: pass\nsc.render.use_persistent_data = True\nsc.render.use_motion_blur = r.get(\'motion_blur\', True); sc.render.motion_blur_shutter = 0.4\nsc.view_settings.view_transform = \'AgX\'\nfor look in (\'AgX - Medium High Contrast\', \'AgX - Punchy\', \'Medium High Contrast\'):\n    try: sc.view_settings.look = look; break\n    except Exception: pass\nsc.render.image_settings.file_format = \'PNG\'\nos.makedirs(r[\'out\'], exist_ok=True)\nif spec.get(\'save_blend\'): bpy.ops.wm.save_as_mainfile(filepath=spec[\'save_blend\'])\nprint(f"escena lista ({time.time() - t0:.0f}s); renderizando {r[\'start\']}-{r[\'end\']} cada {r.get(\'step\', 1)}", flush=True)\nfor f in range(r[\'start\'], r[\'end\'] + 1, r.get(\'step\', 1)):\n    out = os.path.join(r[\'out\'], f"{f:05d}.png")\n    if os.path.exists(out) and not r.get(\'overwrite\'): continue\n    sc.frame_set(f); sc.render.filepath = out; bpy.ops.render.render(write_still=True)\n    print(f"CUADRO {f}", flush=True)\nprint("FIN", flush=True)\n'}
for _n, _s in MODULOS.items():
    open(os.path.join(CODE, _n + '.py'), 'w').write(_s)
print('Módulos listos:', ', '.join(MODULOS))

In [ ]:
#@title 4 · Tus personajes: los busco en Drive, les pongo esqueleto y detecto ojos y boca
import importlib, rig, promo
for m_ in (rig, promo): importlib.reload(m_)
pdir = find_dir(CARPETA_PERSONAJES); MOD = os.path.join(WORK, 'modelos'); os.makedirs(MOD, exist_ok=True)
if not os.path.isdir(pdir): raise FileNotFoundError(f"Sube los .zip de los modelos a {pdir}")
for z in glob.glob(os.path.join(pdir, '**', '*.zip'), recursive=True):
    with zipfile.ZipFile(z) as zf:
        for mem in zf.namelist():                                  # extracción segura (sin rutas raras)
            if mem.endswith('/') or '..' in mem: continue
            dst = os.path.join(MOD, os.path.splitext(os.path.basename(z))[0], os.path.basename(mem))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with zf.open(mem) as src, open(dst, 'wb') as out: out.write(src.read())
for f_ in glob.glob(os.path.join(pdir, '**', '*.*'), recursive=True):
    if f_.lower().endswith(('.obj', '.png', '.jpg', '.mtl')):
        dst = os.path.join(MOD, os.path.basename(os.path.dirname(f_)) or 'sueltos', os.path.basename(f_)); os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst): shutil.copy(f_, dst)
def find_model(key):
    objs = [o for o in glob.glob(os.path.join(MOD, '**', '*.obj'), recursive=True) if key.lower() in o.lower()]
    if not objs: raise FileNotFoundError(f"No encuentro un .obj con «{key}» en {pdir}")
    obj = objs[0]; folder = os.path.dirname(obj); tex = None
    mtl = os.path.splitext(obj)[0] + '.mtl'
    if os.path.exists(mtl):
        for l in open(mtl, errors='ignore'):
            if l.strip().lower().startswith('map_kd'):
                cand = os.path.join(folder, os.path.basename(l.split(maxsplit=1)[1].strip()))
                if os.path.exists(cand): tex = cand
    tex = tex or next(iter(glob.glob(os.path.join(folder, '*.png'))), None)
    if tex is None: raise FileNotFoundError(f"El modelo {obj} no tiene textura .png")
    return obj, tex
AV_FILES = {'A': find_model(MODELO_A), 'B': find_model(MODELO_B)}
AVD, sheets = {}, []
for who, (obj, tex) in AV_FILES.items():
    t_ = cv2.imread(tex, cv2.IMREAD_UNCHANGED)
    if t_.ndim == 2: t_ = cv2.cvtColor(t_, cv2.COLOR_GRAY2BGR)
    AVD[who] = promo.avatar_data(obj, t_)
    img = AVD[who]['face']['front_img'].copy()
    V, VT, G, _ = rig.load_obj(obj); names, info, P = rig.classify(V, G)
    hv = P[info[[g for g, n in names.items() if n == 'Head'][0]]['vi']]; mn = hv.min(0); sc = 600 * 0.9 / (hv.max(0) - mn)[:2].max()
    for k in ('eye_L', 'eye_R', 'mouth'):
        c = AVD[who]['face'][k][0]; x = int((c[0] - mn[0]) * sc + 30); y = int(600 - ((c[1] - mn[1]) * sc + 30)); r = int(AVD[who]['face'][k][1] * sc)
        cv2.circle(img, (x, y), max(4, r), (0, 255, 0) if k != 'mouth' else (0, 0, 255), 3)
    cv2.putText(img, f"{who}: {os.path.basename(obj)}", (10, 30), 0, 0.8, (255, 255, 255), 2); sheets.append(img)
    print(f"{who}: {os.path.basename(obj)} · 15 piezas identificadas · altura {AVD[who]['height']:.1f}")
cv2.imwrite(os.path.join(WORK, 'caras.png'), np.hstack(sheets))
try:
    from IPython.display import Image as IPImage, display; display(IPImage(os.path.join(WORK, 'caras.png'), width=900))
except Exception: pass
print("Verde = ojos (párpados 3D) · rojo = boca 3D que se abre al hablar.")

In [ ]:
#@title 5 · Guion: lo que dicen y lo que hace cada uno (edita las líneas si quieres)
#@markdown A = el del celular · B = el que lanza el comando. `prompt` = lo que la IA de movimiento hace (en inglés).
#@markdown Sets: `blanco`, `negro`, `hdri:plaza`, `hdri:bosque`. Cámaras: `medio_A/B`, `cerca_A/B`, `amplio`, `pov_A/B` (vista desde los ojos de A/B).
BEATS = promo.default_beats(SERVER, DOMINIO)
# Ejemplo para cambiar una línea:  BEATS[0]['linea'] = "¡Jajaja, mira esto!…"
for b in BEATS:
    who = b.get('habla') or '—'
    print(f"[{b['id']:10s}] {b['set']:12s} {b['cam']:8s} {who}: {b.get('linea') or '(acción: ' + (b.get('evento') or '') + ')'}")
    for k in ('A', 'B'): print(f"        {k} → {b[k]['prompt']}")

In [ ]:
#@title 6 · Voces IA de los dos personajes
import soundfile as sf
VOZ_DIR = os.path.join(WORK, 'voces'); os.makedirs(VOZ_DIR, exist_ok=True)
PRON = {"Discord": "Díscord", "server": "sérver", "servers": "sérvers", "The Republic of Furrys": "De Ripáblic of Fúrris", "furros": "furros", "link": "link"}
def para_voz(t):
    for a, b_ in PRON.items(): t = re.sub(r'\b' + re.escape(a) + r'\b', b_, t)
    return t.replace('…', '... ').replace('¿', '').replace('¡', '')
KOKORO = """
import sys, json, numpy as np, soundfile as sf
from kokoro import KPipeline
jobs = json.load(open(sys.argv[1])); p = KPipeline(lang_code='e', repo_id='hexgrad/Kokoro-82M')
for j in jobs:
    ch = [(a.detach().cpu().numpy() if hasattr(a, 'detach') else np.asarray(a)) for _, _, a in p(j['text'], voice=j['voice'], speed=j['speed'])]
    sf.write(j['out'], np.concatenate([np.asarray(a, np.float32) for a in ch]), 24000)
"""
jobs = [{'text': para_voz(b['linea']), 'out': os.path.join(VOZ_DIR, b['id'] + '.wav'), 'voice': VOZ_A if b['habla'] == 'A' else VOZ_B, 'speed': VELOCIDAD_VOZ}
        for b in BEATS if b.get('linea')]
for j in jobs:
    if os.path.exists(j['out']): os.remove(j['out'])
ok = False
if os.path.exists('/content/tts_env/bin/python'):
    open(os.path.join(VOZ_DIR, 'k.py'), 'w').write(KOKORO); json.dump(jobs, open(os.path.join(VOZ_DIR, 'jobs.json'), 'w'), ensure_ascii=False)
    r = subprocess.run(['/content/tts_env/bin/python', os.path.join(VOZ_DIR, 'k.py'), os.path.join(VOZ_DIR, 'jobs.json')], capture_output=True, text=True)
    ok = r.returncode == 0
    if not ok: print("⚠ Kokoro falló, uso Edge:", r.stderr[-300:])
EDGE = {'em_alex': 'es-MX-JorgeNeural', 'em_santa': 'es-ES-AlvaroNeural', 'ef_dora': 'es-MX-DaliaNeural'}
for j in jobs:
    if not os.path.exists(j['out']):
        mp3 = j['out'][:-4] + '.mp3'; rate = f"{int(round((j['speed'] - 1) * 100)):+d}%"
        subprocess.run([sys.executable, '-m', 'edge_tts', '--voice', EDGE[j['voice']], f'--rate={rate}', '--text', j['text'].replace('De Ripáblic of Fúrris', 'The Republic of Furrys'),
                        '--write-media', mp3], capture_output=True)
        if os.path.exists(mp3): subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', mp3, '-ar', '24000', '-ac', '1', j['out']])
VOICES = {}
for b in BEATS:
    p_ = os.path.join(VOZ_DIR, b['id'] + '.wav')
    if b.get('linea') and os.path.exists(p_):
        w, sr = sf.read(p_, dtype='float32'); w = w if w.ndim == 1 else w.mean(1)
        nz = np.where(np.abs(w) > 0.01)[0]
        if len(nz): w = w[max(0, nz[0] - int(0.03 * sr)):nz[-1] + int(0.08 * sr)]       # recorta silencios
        VOICES[b['id']] = (w, sr)
    elif b.get('linea'):
        print(f"⚠ Sin voz para «{b['id']}»: dejo el tiempo de lectura"); n = int(len(b['linea'].split()) / 2.7 * 24000); VOICES[b['id']] = (np.zeros(n, np.float32), 24000)
print(f"🎙️ {len(VOICES)} líneas · {sum(len(w) / sr for w, sr in VOICES.values()):.1f} s de voz")
try:
    from IPython.display import Audio, display; w0, sr0 = VOICES[BEATS[0]['id']]; display(Audio(w0, rate=sr0))
except Exception: pass

In [ ]:
#@title 7 · La IA mueve a los personajes (HY-Motion: texto → movimiento humano real)
import movimiento as mv, director as dr
importlib.reload(mv); importlib.reload(dr)
TOTAL = dr.plan(BEATS, {k: len(w) / sr for k, (w, sr) in VOICES.items()})
print(f"Duración: {TOTAL / FPS:.1f} s ({TOTAL} cuadros)")
HY = '/content/HY-Motion-1.0'; MOT = os.path.join(WORK, 'movimiento'); os.makedirs(MOT, exist_ok=True)
HY_RUN = r'''
import os, sys, json
REPO = sys.argv[1]; os.chdir(REPO); sys.path.insert(0, REPO); os.environ['USE_HF_MODELS'] = '1'
from huggingface_hub import snapshot_download
model = sys.argv[3]
snapshot_download('tencent/HY-Motion-1.0', allow_patterns=[model + '/*'], local_dir=os.path.join(REPO, 'ckpts', 'tencent'))
import torch
from hymotion.utils.gradio_runtime import SimpleRuntime
rt = SimpleRuntime(config_path=os.path.join(REPO, 'ckpts', 'tencent', model, 'config.yml'), ckpt_name='latest.ckpt',
                   load_prompt_engineering=False, load_text_encoder=True).to('cuda').eval()
jobs = json.load(open(sys.argv[2]))
for j in jobs:
    if os.path.exists(os.path.join(j['dir'], j['name'] + '_000.npz')): continue
    try:
        with torch.no_grad():
            rt.generate_motion(j['text'], str(j.get('seed', 42)), j['dur'], 5.0, output_format='dict', output_dir=j['dir'], output_filename=j['name'])
        print('OK', j['name'], flush=True)
    except Exception as e:
        print('FALLO', j['name'], repr(e)[:300], flush=True)
'''
SMPL_REST = rig.SMPL_REST_APPROX
if IA_MOVIMIENTO != 'procedural' and IN_COLAB:
    if not os.path.exists(HY):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Tencent-Hunyuan/HY-Motion-1.0.git', HY])
    if not restore_env('motion_env'):
        print("Instalando HY-Motion (entorno aparte; 1.ª vez ~5-8 min)…")
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/motion_env'], check=True)
        req = [l.strip() for l in open(os.path.join(HY, 'requirements.txt')) if l.strip() and not l.startswith(('#', '--')) and not l.startswith(('fbxsdkpy', 'bitsandbytes'))]
        r = subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/motion_env/bin/python', *req, 'huggingface_hub', 'hf_transfer'], capture_output=True, text=True)
        if r.returncode != 0: print("⚠ HY-Motion no se instaló:", r.stderr[-600:])
        else: save_env('motion_env')
    jt = os.path.join(HY, 'scripts', 'gradio', 'static', 'assets', 'dump_wooden', 'j_template.bin')
    if os.path.exists(jt): SMPL_REST = np.fromfile(jt, np.float32).reshape(-1, 3)[:22].astype(float); print("Esqueleto de referencia de HY-Motion ✓")
    jobs = []
    for b in BEATS:
        for who in ('A', 'B'):
            jobs.append({'text': b[who]['prompt'], 'dur': float(np.clip(b['n'] / FPS + 0.3, 1.0, 12.0)), 'name': f"{b['id']}_{who}", 'dir': MOT, 'seed': 42 + len(jobs)})
    json.dump(jobs, open(os.path.join(MOT, 'jobs.json'), 'w'))
    open(os.path.join(MOT, 'run.py'), 'w').write(HY_RUN)
    env = dict(os.environ, HF_HUB_ENABLE_HF_TRANSFER='1')
    print(f"La IA genera {len(jobs)} movimientos (la 1.ª vez descarga ~22 GB de modelos)…")
    pr = subprocess.Popen(['/content/motion_env/bin/python', os.path.join(MOT, 'run.py'), HY, os.path.join(MOT, 'jobs.json'), IA_MOVIMIENTO],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    log = []
    for line in pr.stdout:
        log.append(line)
        if line.startswith(('OK', 'FALLO')): print(' ', line.strip())
    pr.wait()
    if pr.returncode != 0: print("⚠ HY-Motion terminó con error (uso movimiento de respaldo donde falte):\n", ''.join(log[-15:]))
def get_clip(b, who):
    p_ = os.path.join(MOT, f"{b['id']}_{who}_000.npz")
    if os.path.exists(p_):
        d = np.load(p_); return mv.Clip(d['poses'], d['trans'])
    return mv.procedural(b[who]['kind'], b['n'], seed=abs(hash(b['id'] + who)) % 1000)
n_ia = sum(os.path.exists(os.path.join(MOT, f"{b['id']}_{w}_000.npz")) for b in BEATS for w in 'AB')
print(f"🤖 Movimientos de la IA: {n_ia}/{2 * len(BEATS)}" + ("" if n_ia == 2 * len(BEATS) else " (el resto: respaldo procedural)"))
subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv'], capture_output=False) if IN_COLAB else None

In [ ]:
#@title 8 · Armar la escena: movimiento en tus avatares, mirada, labios, cámaras, silla, flechas, textos 3D y escenarios
from PIL import Image, ImageDraw, ImageFont
import urllib.request
ARR, SPEC, INFOS = promo.assemble(BEATS, VOICES, get_clip, AVD, SMPL_REST)
assert SPEC['total'] == TOTAL
np.savez(os.path.join(WORK, 'datos.npz'), **ARR)
def gfont(fam, w, dst):
    if os.path.exists(dst): return dst
    for q in (f"{fam.replace(' ', '+')}:wght@{w}", fam.replace(' ', '+')):
        try:
            css = urllib.request.urlopen(f"https://fonts.googleapis.com/css2?family={q}", timeout=15).read().decode()
            urllib.request.urlretrieve(re.search(r'url\((https://[^)]+\.ttf)\)', css).group(1), dst); return dst
        except Exception: pass
    return None
FONT_3D = gfont('Anton', 400, os.path.join(WORK, 'Anton.ttf')); FONT_UI = gfont('Inter', 700, os.path.join(WORK, 'Inter.ttf'))
def pil_font(sz):
    try: return ImageFont.truetype(FONT_UI, sz)
    except Exception: return ImageFont.load_default()
# pantalla del celular: un chat genérico con el server "muerto"
im = Image.new('RGB', (360, 720), (49, 51, 56)); d = ImageDraw.Draw(im)
d.rectangle((0, 0, 360, 70), fill=(43, 45, 49)); d.text((20, 22), "# general", font=pil_font(26), fill=(230, 230, 235))
msgs = [("bot", "Bienvenido al server"), ("bot", "Bienvenido al server"), ("???", "hola?"), ("???", "alguien?"), ("bot", "Bienvenido al server")]
for k, (u, m) in enumerate(msgs):
    y = 100 + k * 115; d.ellipse((16, y, 64, y + 48), fill=(88, 101, 242) if u == 'bot' else (120, 120, 130))
    d.text((80, y), u, font=pil_font(22), fill=(240, 240, 240)); d.text((80, y + 32), m, font=pil_font(20), fill=(190, 190, 195))
im.save(os.path.join(WORK, 'pantalla.png'))
# burbuja de chat estilo Roblox con el comando
bub = Image.new('RGBA', (520, 170), (0, 0, 0, 0)); d = ImageDraw.Draw(bub)
d.rounded_rectangle((0, 0, 519, 130), 40, fill=(255, 255, 255, 240)); d.polygon([(240, 130), (280, 130), (260, 168)], fill=(255, 255, 255, 240))
d.text((260, 65), "/silla", font=pil_font(70), fill=(20, 20, 20), anchor='mm'); bub.save(os.path.join(WORK, 'burbuja.png'))
# HDRIs de Poly Haven (CC0)
HDRIS = {}
for key, cands in (('hdri:plaza', HDRI_PLAZA), ('hdri:bosque', HDRI_BOSQUE)):
    for name in [c.strip() for c in cands.split(',') if c.strip()]:
        dst = os.path.join(WORK, f"{name}_4k.hdr")
        if os.path.exists(dst): HDRIS[key] = dst; break
        try:
            j = json.loads(urllib.request.urlopen(f"https://api.polyhaven.com/files/{name}", timeout=20).read())
            url = j['hdri']['4k']['hdr']['url']; urllib.request.urlretrieve(url, dst); HDRIS[key] = dst; print("HDRI", key, "→", name); break
        except Exception as e: print(f"  (no pude bajar el HDRI {name}: {str(e)[:80]})")
J = AVD['A']['J']; tip = (J['_RightHandTip'] - J['RightHand']) * 0.8
SPEC.update({'avatars': {k: {'name': ('A_' if k == 'A' else 'B_') + os.path.splitext(os.path.basename(o))[0], 'obj': o, 'tex': t, 'fur': PELAJE}
                         for k, (o, t) in AV_FILES.items()},
             'npz': os.path.join(WORK, 'datos.npz'), 'hdris': HDRIS, 'font': FONT_3D,
             'phone': {'who': 'A', 'ranges': SPEC['phone_ranges'], 'screen': os.path.join(WORK, 'pantalla.png'), 'tip_local': [float(tip[0]), float(-tip[2]), float(tip[1])]},
             'bubble': {'who': 'B', 'f0': SPEC['bubble_range'][0], 'f1': SPEC['bubble_range'][1], 'img': os.path.join(WORK, 'burbuja.png')}})
print(f"✓ Escena lista: {TOTAL / FPS:.1f} s · silla a {SPEC['chair']['height']:.2f} de alto · {len(SPEC['texts'])} textos 3D · sets: {', '.join(SPEC['sets'])}")

In [ ]:
#@title 9 · Visor: vista previa rápida de cómo va quedando (baja resolución, en vivo)
#@markdown Renderiza un borrador mientras ves los cuadros aparecer aquí abajo; al terminar te deja el video con audio para revisarlo.
CADA_CUANTOS_CUADROS = 2  #@param {type:"slider", min:1, max:6, step:1}
import audio_promo as ap
importlib.reload(ap)
try:
    from IPython.display import display, Image as IPImage, HTML, clear_output
except Exception: display = None
def render_job(tag, w, h, samples, step, fstop=4.0, save_blend=None):
    out = os.path.join(WORK, 'frames_' + tag); os.makedirs(out, exist_ok=True)
    sp = dict(SPEC); sp['render'] = {'w': w, 'h': h, 'samples': samples, 'fps': FPS, 'start': 1, 'end': TOTAL, 'step': step, 'out': out,
                                     'dof': True, 'fstop': fstop, 'motion_blur': tag != 'borrador', 'overwrite': False}
    if save_blend: sp['save_blend'] = save_blend
    p_ = os.path.join(WORK, f'spec_{tag}.json'); json.dump(sp, open(p_, 'w'), default=float)
    pr = subprocess.Popen([sys.executable, os.path.join(CODE, 'job.py'), p_], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=CODE)
    hnd = display(HTML("<i>preparando la escena…</i>"), display_id=True) if display else None
    t0 = time.time(); last = 0; n_tot = len(range(1, TOTAL + 1, step)); done = 0; tail = []
    for line in pr.stdout:
        tail = (tail + [line])[-25:]
        if line.startswith('CUADRO'):
            done += 1; f = int(line.split()[1])
            if hnd and time.time() - last > 4:
                last = time.time(); eta = (time.time() - t0) / done * (n_tot - done)
                img = cv2.imread(os.path.join(out, f"{f:05d}.png"))
                if img is not None:
                    k = 360 / img.shape[0]; ok, buf = cv2.imencode('.jpg', cv2.resize(img, None, fx=k, fy=k))
                    hnd.update(HTML(f"<div style='font-family:sans-serif'><b>{tag}</b> · cuadro {f}/{TOTAL} · {done}/{n_tot} · faltan ~{eta / 60:.1f} min<br>"
                                    f"<img src='data:image/jpeg;base64,{base64.b64encode(buf).decode()}'/></div>"))
        elif line.startswith(('personajes', 'escena', 'Render en')): print(line.strip())
    pr.wait()
    if pr.returncode != 0: raise RuntimeError("El render falló:\n" + ''.join(tail))
    return out
def montaje(frames_dir, out_mp4, w, h, step):
    """Cuadros + subtítulos + audio → MP4 (en los cuadros que no se renderizaron repite el anterior)."""
    audio = ap.mix(BEATS, VOICES, TOTAL, SPEC, VOLUMEN_MUSICA); import soundfile as sf
    wav = os.path.join(WORK, 'mezcla.wav'); sf.write(wav, audio, ap.SR)
    subs = []
    if SUBTITULOS:
        fz = int(56 * w / 1080); fnt = pil_font(fz)
        for b in BEATS:
            if not b.get('linea'): continue
            dur = len(VOICES[b['id']][0]) / VOICES[b['id']][1]
            words = b['linea'].split(); lines, cur = [], ''
            for wd in words:
                t_ = (cur + ' ' + wd).strip()
                if fnt.getlength(t_) > w * 0.84 and cur: lines.append(cur); cur = wd
                else: cur = t_
            lines.append(cur)
            img = Image.new('RGBA', (w, int(fz * 1.35 * len(lines) + 20)), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
            col = (255, 120, 120) if b['habla'] == 'A' else (255, 215, 120)
            for i, l in enumerate(lines):
                d.text((w / 2, 10 + i * fz * 1.35 + fz / 2), l, font=fnt, fill=(255, 255, 255), anchor='mm', stroke_width=max(3, fz // 12), stroke_fill=(0, 0, 0))
            d.rectangle((w * 0.47, 0, w * 0.53, 4), fill=col)
            subs.append((b['voz_f'] + 1, b['voz_f'] + 1 + int(dur * FPS) + 6, np.array(img)))
    enc = ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '18'] if IN_COLAB else ['-c:v', 'libx264', '-crf', '18', '-preset', 'fast']
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{w}x{h}', '-r', str(FPS), '-i', '-', '-i', wav,
           '-map', '0:v', '-map', '1:a', *enc, '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '256k', '-shortest', '-movflags', '+faststart', out_mp4]
    pr = subprocess.Popen(cmd, stdin=subprocess.PIPE); last = None
    for f in range(1, TOTAL + 1):
        fp = os.path.join(frames_dir, f"{f:05d}.png"); img = cv2.imread(fp) if os.path.exists(fp) else None
        if img is None: img = last if last is not None else np.zeros((h, w, 3), np.uint8)
        if img.shape[:2] != (h, w): img = cv2.resize(img, (w, h))
        last = img; fr = img.copy()
        for a, b_, s in subs:
            if a <= f < b_:
                y0 = int(h * 0.76); hh = min(s.shape[0], h - y0); al = s[:hh, :, 3:4] / 255.0
                fr[y0:y0 + hh] = (fr[y0:y0 + hh] * (1 - al) + s[:hh, :, 2::-1] * al).astype(np.uint8)
        pr.stdin.write(fr.tobytes())
    pr.stdin.close(); pr.wait()
    return out_mp4
fd = render_job('borrador', 360, 640, 12, CADA_CUANTOS_CUADROS, fstop=8.0)
prev = montaje(fd, os.path.join(WORK, 'vista_previa.mp4'), 360, 640, CADA_CUANTOS_CUADROS)
shutil.copy(prev, os.path.join(OUT_DIR, 'promo_personajes_vista_previa.mp4'))
if display:
    display(HTML(f"<video width=300 controls src='data:video/mp4;base64,{base64.b64encode(open(prev, 'rb').read()).decode()}'></video>"))
print("Vista previa también en tu Drive:", os.path.join(OUT_DIR, 'promo_personajes_vista_previa.mp4'))

In [ ]:
#@title 10 · Render final en la H100 (Cycles + OptiX) con visor en vivo
W_, H_, S_ = QUAL
blend = os.path.join(OUT_DIR, 'promo_personajes.blend') if GUARDAR_BLEND else None
fd_final = render_job('final', W_, H_, S_, 1, fstop=4.0, save_blend=blend)
if blend: print("Escena de Blender guardada (puedes abrirla en Blender):", blend)

In [ ]:
#@title 11 · Montaje final: subtítulos + voces + música + efectos → MP4 en tu Drive
final = montaje(fd_final, os.path.join(WORK, 'promo_personajes_ia.mp4'), W_, H_, 1)
dst = os.path.join(OUT_DIR, 'promo_personajes_ia.mp4'); shutil.copy(final, dst)
print("✓ Video final:", dst, f"({TOTAL / FPS:.1f} s, {W_}x{H_})")